# CASMI 2026 — scoring repair 2.0.2

**Default:** export the inherited neural control, verify the saved `submission.csv`, and keep experimental reranking optional.

Your attached 2.0.1 notebook completed its visible run in 12,261.71 seconds and produced 400 rows. You reported **Submission Scoring Error** on the competition. The visible notebook log contains no grader traceback or offending prediction, so the exact grading failure is not established by those files. A successful visible run is not proof of accepted hidden scoring.

This release addresses concrete output risks:

- Creates the legacy sample template from the actual runtime test IDs; an unchanged visible sample cannot introduce stale IDs.
- Validates IDs, schema, nulls, SMILES and the 25-candidate limit after saving and rereading the actual CSV.
- Prints the exact selected path and a submission certificate; preserves other CSVs in a ZIP archive.
- Saves a verified control before any optional challenger stage. New-stage errors cannot remove it.
- Defaults to the inherited weight-0.25 neural control while the submission issue is isolated. No unvalidated challenger policy is enabled by default.

## Run and submit

1. Import this notebook, attach the same six original datasets plus the competition, and use Python 3.12 with T4 GPU and Internet OFF.
2. Save Version → **Save & Run All**. Use the newly completed version.
3. Select **`submission.csv` at the notebook output root** for the competition submission. Do not select a diagnostic archive, sample template, reference file, or CSV downloaded from a visible example run.
4. The final log prints `SELECT THIS COMPETITION OUTPUT: submission.csv`, plus current-test row count, header, candidate counts and SHA-256.

This notebook contains all refactor source and the original attributed ensemble. Full hidden grading and any leaderboard improvement remain unverified. Locally `auto` performs a data audit; on Kaggle it performs full inference. The recorded local outputs below are audit/engineering results, not a competition score.


## Fix 2.0.1 — tied reference IDs and recovery

The supplied Kaggle log failed in `heapq.heappush` after about 10,061 seconds. The baseline and neural control had completed for all 400 molecules. The reference reservoir previously stored tuples ending in a row containing NumPy arrays. Equal priority/ID/key values caused tuple comparison to reach those arrays and raise `ValueError: The truth value of an array with more than one element is ambiguous`.

`ReferenceEntry` now excludes its structure key and array-bearing row from comparisons. Both heap insertion and replacement compare only scalar fields. RDKit kekulization/tautomer messages earlier in that log were warnings; this heap comparison was the fatal exception.

Resume is enabled by default when completed checkpoint files are still present in the same session. It verifies the pinned baseline source, full query coverage, and equality of every original query column (including peak arrays and CE). A mismatched or partial checkpoint is rejected. If files from the failed session are unavailable, the notebook runs the baseline normally in a fresh session.

The new code is a bug fix. No leaderboard gain is claimed.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, importlib.util
RUN_MODE = "auto"  # auto / submission / audit / smoke
IS_KAGGLE = Path("/kaggle/input").is_dir()
MODE = ("submission" if IS_KAGGLE else "audit") if RUN_MODE == "auto" else RUN_MODE
ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
PACKAGE = ROOT / "casmi_refactor"
PACKAGE.mkdir(parents=True, exist_ok=True)
LEGACY = ROOT / "casmi_legacy"
LEGACY.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
print("Mode:", MODE)
print("Python:", sys.version.split()[0])
print("Full pretrained inference has not been executed in the delivered local outputs.")


## Complete modular Python source

Each cell writes one readable, importable module. All source is included; no online downloads are performed by these cells.

In [ ]:
# Module: casmi_refactor/__init__.py
module_source = r'''"""CASMI retrieval overhaul: audited compatibility inference and trainable upgrades."""
__version__ = "2.0.2"
'''
(PACKAGE / '__init__.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/__main__.py
module_source = r'''"""CLI for audit, fold preparation, checkpoint training and offline Kaggle inference."""
import argparse
from pathlib import Path
import json
from .config import Config


def main():
    p = argparse.ArgumentParser(description=__doc__)
    sub = p.add_subparsers(dest="command", required=True)
    a = sub.add_parser("audit"); a.add_argument("--test", required=True); a.add_argument("--sample", required=True); a.add_argument("--output", required=True)
    a = sub.add_parser("split"); a.add_argument("--train", required=True); a.add_argument("--output", required=True); a.add_argument("--folds", type=int, default=5)
    a = sub.add_parser("prepare"); a.add_argument("--train", required=True); a.add_argument("--split", required=True); a.add_argument("--output", required=True)
    a = sub.add_parser("train"); a.add_argument("--store", required=True); a.add_argument("--output", required=True)
    a.add_argument("--train-folds", nargs="+", type=int, required=True); a.add_argument("--dev-folds", nargs="+", type=int, required=True)
    a.add_argument("--test-folds", nargs="+", type=int, required=True); a.add_argument("--epochs", type=int, default=15)
    a = sub.add_parser("infer"); a.add_argument("--legacy", required=True); a.add_argument("--config", required=True)
    args = p.parse_args()
    if args.command == "audit":
        from .audit import inspect_inputs
        print(json.dumps(inspect_inputs(args.test,args.sample,args.output)[0],indent=2))
    elif args.command == "split":
        import pandas as pd
        import pyarrow.parquet as pq
        from .validation import split_manifest
        unique = set()
        for b in pq.ParquetFile(args.train).iter_batches(columns=["normalized_smiles"],batch_size=65536):
            unique.update(b.column(0).to_pylist())
        result = split_manifest(pd.DataFrame({"smiles": sorted(s for s in unique if s)}),args.folds)
        result.to_parquet(args.output,index=False)
        print("Fold structures:", result.groupby("fold").size().to_dict())
    elif args.command == "prepare":
        import pandas as pd
        from .training import prepare_store
        print(prepare_store(args.train,pd.read_parquet(args.split),args.output))
    elif args.command == "train":
        from .training import fit_dual
        fit_dual(args.store,args.output,args.train_folds,args.dev_folds,args.test_folds,epochs=args.epochs)
    elif args.command == "infer":
        from .pipeline import run_submission
        run_submission(Config(**json.loads(Path(args.config).read_text())),args.legacy)


if __name__=="__main__":
    main()
'''
(PACKAGE / '__main__.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/audit.py
module_source = r'''"""Read-only audit of the supplied visible test; never treats it as independent validation."""
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
from rdkit import rdBase
from .config import Config
from .spectra import from_row, mass_consensus
from .runtime import sha256


def inspect_inputs(test_path, sample_path, destination, cfg=Config()):
    """Summarize actual spectra, mass consistency and ID coverage with honest limitations."""
    test = pd.read_parquet(test_path)
    sample = pd.read_csv(sample_path, dtype=str, keep_default_na=False)
    required = {"molecule_id", "spectrum_id", "ms2_mzs", "ms2_normalized_intensities",
                "adduct", "precursor_mz", "ionization_mode"}
    if required-set(test):
        raise ValueError(f"Missing columns {required-set(test)}")
    cleaned, mismatch, empty = [], 0, 0
    for row in test.itertuples(index=False):
        mismatch += int(len(row.ms2_mzs) != len(row.ms2_normalized_intensities))
        sp = from_row(row, cfg)
        cleaned.append(len(sp.mz)); empty += int(not len(sp.mz))
    consensus = []
    for mid, g in test.groupby("molecule_id", sort=False):
        mass, spread, alternatives = mass_consensus(g)
        consensus.append(dict(molecule_id=str(mid), neutral_mass=mass, mass_mad_ppm=spread,
            spectra=len(g), adducts=g.adduct.nunique(), hypotheses=len(alternatives)))
    consensus = pd.DataFrame(consensus)
    report = dict(python=platform.python_version(), rdkit=rdBase.rdkitVersion,
        test_sha256=sha256(test_path), sample_sha256=sha256(sample_path),
        spectra=len(test), molecules=int(test.molecule_id.nunique()),
        spectrum_id_unique=bool(test.spectrum_id.is_unique),
        sample_id_coverage=bool(set(sample.molecule_id)==set(test.molecule_id)),
        aligned_array_failures=mismatch, empty_after_structural_cleaning=empty,
        cleaned_peak_quantiles=dict(zip(["min","p25","median","p75","max"],
            np.quantile(cleaned, [0,.25,.5,.75,1]).tolist())),
        adduct_counts={str(k): int(v) for k,v in test.adduct.value_counts().items()},
        mass_mad_ppm_median=float(consensus.mass_mad_ppm.median()),
        mass_mad_ppm_max=float(consensus.mass_mad_ppm.max()),
        independent_accuracy=None, leaderboard_score=None,
        limitation="Visible test has no labels here and overlaps competition training; this is a data/engineering audit only.")
    dest = Path(destination); dest.mkdir(parents=True, exist_ok=True)
    (dest/"input_audit.json").write_text(json.dumps(report, indent=2, allow_nan=False))
    consensus.to_csv(dest/"mass_consistency.csv", index=False)
    return report, consensus
'''
(PACKAGE / 'audit.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/chemistry.py
module_source = r'''"""Pinned metric identity and chemistry targets; no destructive salt neutralization."""
from functools import lru_cache
import numpy as np
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

METRIC_RDKIT = "2026.03.3"
_TAUT = rdMolStandardize.TautomerEnumerator()
_MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)


def assert_metric_version():
    """Inference must match the supplied metric version exactly, including patch release."""
    if rdBase.rdkitVersion != METRIC_RDKIT:
        raise RuntimeError(f"Need RDKit {METRIC_RDKIT}; got {rdBase.rdkitVersion}")


@lru_cache(maxsize=100000)
def identity(smiles):
    """Mirror the supplied notebook's stereo-free tautomer InChIKey14 implementation.

    This preserves its enumerator defaults; no custom limits, uncharging, largest-fragment
    selection or formula gating are inserted into the metric. Verify against any updated
    organizer metric before changing the competition's pinned environment.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        Chem.RemoveStereochemistry(mol)
        key = Chem.MolToInchiKey(_TAUT.Canonicalize(mol))
        return key[:14] if key else None
    except (ValueError, RuntimeError, TypeError):
        return None


@lru_cache(maxsize=50000)
def properties(smiles):
    """Return mass/formula/validity flags without discarding charged or unusual molecules."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or identity(smiles) is None:
        return None
    return dict(metric_key=identity(smiles), exact_mass=rdMolDescriptors.CalcExactMolWt(mol),
                formula=rdMolDescriptors.CalcMolFormula(mol),
                charge=Chem.GetFormalCharge(mol), components=len(Chem.GetMolFrags(mol)),
                heavy_atoms=mol.GetNumHeavyAtoms())


@lru_cache(maxsize=50000)
def fingerprint(smiles):
    """Morgan2048 + MACCS167 + 32 explicit SMARTS targets (2247 bits).

    These are open chemical targets, not the proprietary CSI:FingerID bit dictionary.
    Every checkpoint records this schema so incompatible target banks cannot be mixed.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")
    m = np.zeros(2048, dtype=np.uint8)
    k = np.zeros(167, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(_MORGAN.GetFingerprint(mol), m)
    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), k)
    extra = np.array([mol.HasSubstructMatch(p) for p in SMARTS], dtype=np.uint8)
    return np.concatenate([m, k, extra])


SMARTS_TEXT = ["[OX2H]", "[NX3;H1,H2]", "C(=O)[OX2H1]", "C(=O)N", "C(=O)O[#6]",
    "[#6][CX3](=O)[#6]", "[CX3H1](=O)", "c1ccccc1", "[nH]", "n", "S(=O)(=O)",
    "[SX2H]", "P(=O)(O)O", "C#N", "C#C", "[F]", "[Cl]", "[Br]", "[I]", "[N+](=O)[O-]",
    "[OX2r5]", "[OX2r6]", "[NX3r5]", "[NX3r6]", "[CX3]=[CX3]", "[CX4H3]",
    "[CX4H2][OX2H]", "[OX2][CX4][OX2]", "[NX4+]", "[O-]", "[CX3](=O)[CX3](=O)", "[SX2][#6]"]
SMARTS = [Chem.MolFromSmarts(s) for s in SMARTS_TEXT]
FP_SCHEMA = "morgan2-2048+maccs167+smarts32-v1"
FP_BITS = 2247


def group_key(smiles):
    """Bemis–Murcko scaffold; acyclic compounds share a conservative explicit group."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid label in split manifest")
    Chem.RemoveStereochemistry(mol)
    sc = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    return sc or "ACYCLIC"


def unique_candidates(smiles, limit=25):
    """Stable metric-equivalence deduplication; invalid predictions occupy no rank."""
    out, seen = [], set()
    for s in smiles:
        s = str(s).strip()
        key = identity(s)
        if key is not None and key not in seen:
            seen.add(key)
            out.append(s)
            if len(out) == limit:
                break
    return out
'''
(PACKAGE / 'chemistry.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/config.py
module_source = r'''"""Central configuration. Submission inference never uses sample SMILES as predictions."""
from dataclasses import dataclass, asdict
from pathlib import Path
import json


@dataclass(frozen=True)
class Config:
    mode: str = "auto"  # auto = full inference on Kaggle, audit elsewhere
    input_dir: str = ""
    input_root: str = "/kaggle/input"
    output_dir: str = "/kaggle/working"
    seed: int = 2026
    max_guesses: int = 25
    pool_limit: int = 192
    floor: float = 0.002
    max_peaks: int = 160
    window_da: float = 50.0
    window_peaks: int = 10
    fragment_da: float = 0.005
    fragment_ppm: float = 10.0
    mass_ppm: float = 15.0
    fallback_ppm: float = 35.0
    precursor_bias_ppm: float = 0.0  # Set only from independent lock-mass evidence.
    library_batch: int = 16384
    max_library_spectra: int = 1200  # per query; deterministic balanced reservoir
    library_rescue: int = 40
    neural_weight: float = 0.25  # inherited experiment; unvalidated on new union
    overflow_weight: float = 0.12
    library_weight: float = 0.20
    dual_weight: float = 0.15  # used only when a separately trained checkpoint is supplied
    rrf_offset: float = 3.0
    max_new_in_top25: int = 5
    protect_library_top1: float = 0.97
    baseline_timeout_s: int = 27000
    neural_timeout_s: int = 1800
    run_budget_s: int = 30600  # 8.5 h; leaves a platform shutdown margin
    policy_file: str = ""  # optional policy selected with nested grouped validation
    ranker_file: str = ""  # optional fold-safe ranker checkpoint
    dual_checkpoint: str = ""  # optional separately trained Transformer
    resume_completed_baseline: bool = True  # verify exact query/source before reuse
    final_submission: str = "control"  # opt into "challenger" only after grading is restored

    def validate(self):
        """Reject invalid settings before running costly external engines."""
        if self.mode not in {"auto", "audit", "submission", "smoke"}:
            raise ValueError("mode must be auto/audit/submission/smoke")
        if self.final_submission not in {"control", "challenger"}:
            raise ValueError("final_submission must be control/challenger")
        if not 1 <= self.max_guesses <= 25 or self.pool_limit < self.max_guesses:
            raise ValueError("Invalid candidate limits")
        if not 0 <= self.floor < 1 or self.fallback_ppm < self.mass_ppm:
            raise ValueError("Invalid preprocessing tolerances")
        if self.baseline_timeout_s + self.neural_timeout_s >= self.run_budget_s:
            raise ValueError("Reserve time for final ranking and validation")
        return self

    def dump(self, path):
        Path(path).write_text(json.dumps(asdict(self), indent=2))


def locate_competition(cfg):
    """Resolve only explicit official mount variants, never arbitrary recursive matches."""
    roots = [Path(cfg.input_dir)] if cfg.input_dir else [
        Path(cfg.input_root) / "competitions/enveda-CASMI26-molecule-id-mass-spectra",
        Path(cfg.input_root) / "enveda-CASMI26-molecule-id-mass-spectra",
    ]
    found = [p for p in roots if (p / "test.parquet").is_file()]
    if len(found) != 1:
        raise FileNotFoundError(f"Expected one competition directory; found {found}")
    return found[0].resolve()
'''
(PACKAGE / 'config.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/legacy_runner.py
module_source = r'''"""Isolate the supplied ensemble, force full coverage and export candidate overflow.

The notebook source is hash-pinned and distributed with its original attribution. No
pretrained featurizer, simulator, candidate generator or baseline ranking rule is changed.
Only the visible-test smoke switch is overridden in full submission mode.
"""
from pathlib import Path
import json
import os
import sys
from .runtime import sha256


def export_union(scope, root):
    """Export a candidate's earliest rank in each channel and its available simulators."""
    import pandas as pd
    import numpy as np
    from .chemistry import identity
    baseline = pd.read_csv(root/"submission.csv", dtype=str, keep_default_na=False)
    final_map = dict(zip(baseline.molecule_id, baseline.smiles.str.split(";")))
    processed = set(scope["te"].molecule_id.astype(str))
    rows = []
    for mid in sorted(processed):
        base = scope.get("BASE", {}).get(mid, ([], [], 0.))
        channels = {
            "baseline": final_map.get(mid, []),
            "main": scope.get("BASE_TAIL_CANDIDATES", {}).get(mid, list(base[0])),
            "prefusion": scope.get("PREFUSION_LISTS", {}).get(mid, []),
            "engine2": scope.get("ENG", {}).get(mid, {}).get("smiles", []),
            "pubchem": scope.get("PC", {}).get(mid, {}).get("pc", []),
            "overflow": scope.get("FUSION_OVERFLOW", {}).get(mid, []),
        }
        candidates = {}
        for name, smiles in channels.items():
            # Metric duplicates get ONE channel vote and compact unique ranks.
            seen = set()
            for s in smiles:
                k = identity(s)
                if k is None or k in seen:
                    continue
                seen.add(k)
                rec = candidates.setdefault(k, dict(molecule_id=mid, metric_key=k, smiles=s,
                    processed=True, library_max=float(base[2]) if len(base)>2 else 0.))
                rec[name+"_rank"] = len(seen)
        ice = scope.get("ICE_SCORES", {}).get(mid, {}) or {}
        gl = scope["gl_of"](mid) or {}
        # Normalize score dictionaries once; their native keys vary across public engines.
        maps = []
        for scores in [ice, gl]:
            m = {}
            for k, v in scores.items():
                try:
                    if v is None or not np.isfinite(float(v)):
                        continue
                    key = k if isinstance(k, str) and len(k)==14 and k.isalpha() else identity(k)
                    if key:
                        m[key] = max(m.get(key, -np.inf), float(v))
                except (TypeError, ValueError):
                    continue
            maps.append(m)
        for key, row in candidates.items():
            row["ice"], row["gl"] = maps[0].get(key), maps[1].get(key)
            for col in ["baseline", "main", "engine2", "pubchem", "prefusion", "overflow"]:
                row.setdefault(col+"_rank", 0)
            rows.append(row)
    pd.DataFrame(rows).to_parquet(root/"union_evidence.parquet", index=False)
    print("EXPORTED CANDIDATE UNION", len(rows), "rows", flush=True)


def main(source, expected, output, full="1"):
    from IPython.display import display
    root, source = Path(output), Path(source)
    if sha256(source) != expected:
        raise ValueError("Pinned baseline source differs")
    if (root/"work/spec_meta.parquet").exists() or (root/"submission.csv").exists():
        raise RuntimeError("Run legacy engines in a fresh Kaggle session")
    os.chdir(root)
    sys.path.insert(0, str(source.parent))
    scope = {"__name__": "__main__", "display": display}
    for i, cell in enumerate(json.loads(source.read_text())["cells"]):
        if cell["cell_type"] != "code":
            continue
        code = "".join(cell["source"])
        if i == 30 and full == "1":
            from .output_contract import make_runtime_competition
            runtime = make_runtime_competition(scope["COMP"], root/"runtime_competition")
            scope["COMP"] = str(runtime)
            if "SAMPLE_PATH" in scope:
                scope["SAMPLE_PATH"] = runtime/"sample_submission.csv"
            # Keep the same variable contract used by inherited receipts and assertions.
            code = "IS_RERUN=True\nSMOKE_N=0\nprint('FULL COVERAGE: all actual test molecules')\n"
        print(f"LEGACY CELL {i} START", flush=True)
        exec(compile(code, f"legacy_cell_{i}", "exec"), scope)
    from export_evidence import export
    export(scope, root)  # preserves the corrected pre-neural control and query file
    export_union(scope, root)
    info = dict(source_sha256=expected, full_coverage_requested=full=="1",
                processed_molecules=int(scope["te"].molecule_id.nunique()),
                adaptation="cell30 full coverage with actual-test ID template" if full=="1" else "original smoke switch")
    (root/"execution_receipt.json").write_text(json.dumps(info, indent=2))


if __name__ == "__main__":
    main(*sys.argv[1:])
'''
(PACKAGE / 'legacy_runner.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/library.py
module_source = r'''"""Streaming, mass-gated reference rescue; no dense all-library spectral matrix."""
from collections import defaultdict
from dataclasses import dataclass, field
import hashlib
import heapq
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from .chemistry import identity
from .spectra import ADDUCTS, from_row, mass_consensus, compare


@dataclass(order=True)
class ReferenceEntry:
    """Order reservoir entries by scalar priority/ID, never by array-bearing rows.

    Repeated spectrum IDs intentionally have equal sampling priority. Excluding the
    payload from comparisons makes both heap insertion and replacement safe on ties.
    """
    priority: float
    spectrum_id: str
    key: str = field(compare=False)
    row: object = field(compare=False, repr=False)


def stream_references(train_path, queries, cfg, excluded_keys=(), excluded_scaffolds=(), deadline=None):
    """Scan Arrow batches, materializing peak lists only for relevant neutral masses.

    A deterministic platform-prioritized reservoir caps query-specific comparison work. Its sampling
    rate is reported: a capped rescue channel is approximate and must not claim full recall.
    Both key and scaffold exclusions apply BEFORE selecting references in held-out replay.
    """
    from .chemistry import group_key
    contexts = []
    for mid, g in queries.groupby("molecule_id", sort=False):
        _, _, hypotheses = mass_consensus(g, cfg.precursor_bias_ppm)
        contexts.extend((m, str(mid)) for m in hypotheses if m > 0)
    contexts = sorted(set(contexts))
    masses = np.array([x[0] for x in contexts])
    mids = [x[1] for x in contexts]
    if not len(masses):
        return {}, {"rows_scanned": 0, "mass_matches": {}}
    blocked, blocked_sc = set(excluded_keys), set(excluded_scaffolds)
    pf = pq.ParquetFile(train_path)
    desired = ["spectrum_id", "normalized_smiles", "inchikey14", "precursor_mz", "adduct",
        "ionization_mode", "instrument_type", "collision_energy_ev", "base_peak_intensity",
        "ms2_mzs", "ms2_normalized_intensities", "ingest_lib", "precursor_error_ppm"]
    columns = [c for c in desired if c in pf.schema_arrow.names]
    heaps, matches = defaultdict(list), defaultdict(int)
    scanned = 0
    for batch in pf.iter_batches(batch_size=cfg.library_batch, columns=columns):
        if deadline is not None and time.monotonic() > deadline:
            raise TimeoutError("Reference rescue exceeded the reserved run budget")
        prec = batch.column(batch.schema.get_field_index("precursor_mz")).to_numpy(zero_copy_only=False)
        ads = batch.column(batch.schema.get_field_index("adduct")).to_pylist()
        nm = np.array([float(p)-ADDUCTS[a] if a in ADDUCTS and p is not None else np.nan
                       for p, a in zip(prec, ads)])
        scanned += len(batch)
        positions = np.searchsorted(masses, nm)
        left = np.clip(positions-1, 0, len(masses)-1)
        right = np.clip(positions, 0, len(masses)-1)
        delta = np.minimum(abs(nm-masses[left]), abs(nm-masses[right]))
        keep = np.flatnonzero(np.isfinite(nm) & (delta <= nm*cfg.fallback_ppm*1e-6))
        if not len(keep):
            continue
        # Only these rows turn their nested Arrow peak arrays into Python/NumPy objects.
        table = batch.take(keep).to_pandas()
        for row, neutral in zip(table.itertuples(index=False), nm[keep]):
            rawkey = str(getattr(row, "inchikey14", ""))
            if rawkey in blocked:
                continue
            # A large labelled precursor error is a label-quality failure, not a useful decoy.
            err = getattr(row, "precursor_error_ppm", None)
            if err is not None and np.isfinite(err) and abs(err) > cfg.fallback_ppm:
                continue
            key = identity(row.normalized_smiles)
            if key is None or key in blocked:
                continue
            if blocked_sc and group_key(row.normalized_smiles) in blocked_sc:
                continue
            tol = neutral*cfg.fallback_ppm*1e-6
            lo, hi = np.searchsorted(masses, [neutral-tol, neutral+tol])
            destinations = set(mids[lo:hi])
            # Stable, dataset-order-independent sampling, slightly favoring platform match.
            sid = str(getattr(row, "spectrum_id", row.normalized_smiles))
            priority = int.from_bytes(hashlib.sha256(sid.encode()).digest()[:8], "big") / 2**64
            if "tims" in str(row.instrument_type).lower():
                priority += .25
            for mid in destinations:
                matches[mid] += 1
                item = ReferenceEntry(priority, sid, key, row)
                h = heaps[mid]
                if len(h) < cfg.max_library_spectra:
                    heapq.heappush(h, item)
                elif item > h[0]:
                    heapq.heapreplace(h, item)
    return heaps, dict(rows_scanned=scanned, mass_matches=dict(matches),
                       retained={m: len(h) for m, h in heaps.items()},
                       approximate_if_capped=True)


def retrieve(train_path, queries, cfg, excluded_keys=(), excluded_scaffolds=(), deadline=None):
    """Aggregate per-query-spectrum maxima so repeated library spectra cannot cast votes."""
    refs, report = stream_references(train_path, queries, cfg, excluded_keys, excluded_scaffolds, deadline)
    groups = {str(m): g for m, g in queries.groupby("molecule_id", sort=False)}
    result = []
    for mid, heap in refs.items():
        qspecs = [from_row(r, cfg) for r in groups[mid].itertuples(index=False)]
        weights = np.array([q.quality for q in qspecs])
        weights /= max(weights.sum(), 1e-12)
        candidates = {}
        for entry in heap:
            key, row = entry.key, entry.row
            ref = from_row(row, cfg)
            if not len(ref.mz):
                continue
            rec = candidates.setdefault(key, dict(smiles=row.normalized_smiles,
                per_spec=np.zeros(len(qspecs)), library_entropy=0., loss_entropy=0.,
                sqrt_cos=0., matched=0., same_adduct=0., same_instrument=0., ce_delta=1000.))
            for i, query in enumerate(qspecs):
                ev = compare(query, ref, cfg)
                rec["per_spec"][i] = max(rec["per_spec"][i], ev["library_score"])
                for name, source in [("library_entropy", "entropy"), ("loss_entropy", "loss_entropy"),
                    ("sqrt_cos", "sqrt_cos"), ("matched", "matched"),
                    ("same_adduct", "same_adduct"), ("same_instrument", "same_instrument")]:
                    rec[name] = max(rec[name], ev[source])
                rec["ce_delta"] = min(rec["ce_delta"], ev["ce_delta"])
        for key, rec in candidates.items():
            v = rec.pop("per_spec")
            rec.update(molecule_id=mid, metric_key=key,
                library_score=.65*float(weights@v)+.35*float(v.max(initial=0)),
                library_coverage=float(weights@(v > .4)))
            result.append(rec)
    frame = pd.DataFrame(result)
    if len(frame):
        frame = frame.sort_values(["molecule_id", "library_score", "metric_key"],
            ascending=[True, False, True], kind="stable")
        frame["library_rank"] = frame.groupby("molecule_id").cumcount()+1
        frame = frame[frame.library_rank <= cfg.library_rescue].reset_index(drop=True)
    report["rescued_rows"] = len(frame)
    return frame, report
'''
(PACKAGE / 'library.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/pipeline.py
module_source = r'''"""Offline submission orchestration around the supplied public ensemble."""
from pathlib import Path
from dataclasses import asdict
import json
import os
import sys
import time
import pandas as pd
from .config import locate_competition
from .runtime import bootstrap_rdkit, run_logged, sha256


def run_submission(cfg, legacy_dir):
    """Run one full baseline, broaden its union and write validated submission.csv.

    An exact inherited final-25 neural rerank is saved as submission_control.csv.
    The challenger adds pre-truncation candidates and reference evidence. Neither its
    default weights nor its score are claimed validated until independent replay exists.
    """
    cfg.validate()
    comp = locate_competition(cfg)
    root, legacy = Path(cfg.output_dir).resolve(), Path(legacy_dir).resolve()
    root.mkdir(parents=True, exist_ok=True)
    if str(root) != "/kaggle/working":
        raise ValueError("Inherited engines have absolute paths; use /kaggle/working on Kaggle")
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("The supplied offline engines require the Kaggle Python 3.12 image")
    if not (comp/"train.parquet").is_file():
        raise FileNotFoundError("Attach competition train.parquet and the original model datasets")
    pins = json.loads((legacy/"pins.json").read_text())
    started = time.monotonic()
    bootstrap_rdkit(legacy, root)
    from .chemistry import assert_metric_version
    assert_metric_version()
    from .ranking import (build_union, rank_candidates, apply_policy, LearnedRanker,
                          write_submission, validate_submission)
    from .output_contract import (export_verified, final_certificate, restore_resume_csvs,
                                  with_control_fallback)
    cfg = apply_policy(cfg)
    cfg.dump(root/"refactor_config.json")
    sys.path.insert(0, str(legacy))
    from bundle import find_bundle
    bundle = find_bundle(cfg.input_root, root, pins["MODEL_PINS"])
    # Ensure child imports the visible refactor modules, regardless of current working folder.
    package_root = str(Path(__file__).resolve().parents[1])
    env = {"PYTHONPATH": package_root+os.pathsep+os.environ.get("PYTHONPATH", "")}
    test = pd.read_parquet(comp/"test.parquet")
    sample = pd.read_csv(comp/"sample_submission.csv", dtype=str) if (comp/"sample_submission.csv").is_file() else None
    if cfg.resume_completed_baseline:
        restore_resume_csvs(root)
    resume = (cfg.mode != "smoke" and cfg.resume_completed_baseline and
              (root/"submission_control.csv").is_file())
    if resume:
        from .checkpoints import validate_completed_baseline
        receipt = validate_completed_baseline(root, test, pins["FROZEN_SHA256"])
        print("RESUME: verified exact query/source; reusing completed baseline and control scoring", flush=True)
    else:
        if (root/"submission.csv").exists():
            raise RuntimeError("A submission already exists without a verified resume checkpoint; use a fresh session")
        run_logged([sys.executable, "-m", "casmi_refactor.legacy_runner", legacy/"frozen_0465.ipynb",
            pins["FROZEN_SHA256"], root, "0" if cfg.mode=="smoke" else "1"],
            root/"baseline_pipeline.log", root, cfg.baseline_timeout_s, env)
        receipt = json.loads((root/"execution_receipt.json").read_text())
        # The exact final-25 neural control runs first, preserving the user's experiment.
        run_logged([sys.executable, "-m", "casmi_refactor.scoring", bundle, "final_model.pt",
            root/"frozen_candidate_evidence.parquet", root/"effective_query_spectra.parquet",
            root/"control_neural.parquet", legacy, "cuda"], root/"control_neural.log", root,
            cfg.neural_timeout_s//2, env)
        from submission import apply
        apply(root/"control_neural.parquet", root/"submission_baseline0465.csv",
              root/"submission_control.csv", weight=.25, offset=3.)
    if cfg.mode != "smoke" and receipt["processed_molecules"] != test.molecule_id.nunique():
        raise RuntimeError("Baseline did not process every actual test molecule")
    control = pd.read_csv(root/"submission_control.csv", dtype=str, keep_default_na=False)
    if cfg.mode == "smoke":
        # Smoke outputs are never left under the platform's submit-ready filename.
        (root/"submission.csv").rename(root/"legacy_smoke_do_not_submit.csv")
        print("SMOKE ONLY: inherited placeholder rows retained in control; no submission.csv exported")
        return dict(mode="smoke", leaderboard_score=None, processed=receipt["processed_molecules"])
    validate_submission(control, test.molecule_id.unique())
    # Preserve the original output separately, then establish a usable validated control.
    # Every optional challenger operation starts with a complete submission already saved.
    if (root/"submission.csv").exists() and not resume:
        previous = "submission_previous.csv" if resume else "legacy_pre_neural.csv"
        (root/"submission.csv").replace(root/previous)
    export_verified(control, test, root/"submission.csv", sample)
    if cfg.final_submission == "control":
        report = final_certificate(root, test, "control")
        report.update(resumed_completed_baseline=resume,
            runtime_s=round(time.monotonic()-started, 2),
            baseline_source_sha256=pins["FROZEN_SHA256"], measured_improvement=None)
        (root/"refactor_report.json").write_text(json.dumps(report, indent=2))
        return report
    return with_control_fallback(
        lambda: run_challenger(cfg, root, comp, legacy, bundle, env, test, sample, pins, resume, started),
        root, control, test, sample)


def run_challenger(cfg, root, comp, legacy, bundle, env, test, sample, pins, resume, started):
    """Optional experimental branch; failure cannot remove the validated control CSV."""
    from .library import retrieve
    from .ranking import build_union, rank_candidates, LearnedRanker, write_submission
    from .output_contract import export_verified, final_certificate
    query = pd.read_parquet(root/"effective_query_spectra.parquet")
    union = pd.read_parquet(root/"union_evidence.parquet")
    reserved = time.monotonic()+max(1, cfg.run_budget_s-(time.monotonic()-started)-cfg.neural_timeout_s//2-120)
    rescued, lib_report = retrieve(comp/"train.parquet", query, cfg, deadline=reserved)
    rescued.to_parquet(root/"library_rescue.parquet", index=False)
    union = build_union(union, rescued, query, cfg)
    union.to_parquet(root/"expanded_union.parquet", index=False)
    # Reuse the trained featurizer unchanged. The model can now see rescued/overflow structures.
    run_logged([sys.executable, "-m", "casmi_refactor.scoring", bundle, "final_model.pt",
        root/"expanded_union.parquet", root/"effective_query_spectra.parquet",
        root/"union_neural.parquet", legacy, "cuda"], root/"union_neural.log", root,
        cfg.neural_timeout_s//2, env)
    scored = pd.read_parquet(root/"union_neural.parquet")
    if cfg.dual_checkpoint:
        from .training import predict_candidates
        scored = predict_candidates(cfg.dual_checkpoint, query, scored, cfg)
    ranker = LearnedRanker(cfg.ranker_file) if cfg.ranker_file else None
    ranked, audit = rank_candidates(scored, cfg, ranker)
    ranked.to_parquet(root/"candidate_scores.parquet", index=False)
    audit.to_csv(root/"rerank_audit.csv", index=False)
    if time.monotonic()-started > cfg.run_budget_s:
        raise TimeoutError("Overall run budget exhausted; inspect stage logs")
    output = write_submission(ranked, test, root/"challenger_candidate.csv")
    export_verified(output, test, root/"submission.csv", sample)
    report = dict(mode="submission", molecules=len(output), candidate_union=len(scored),
        resumed_completed_baseline=resume,
        changed_top1=int(audit.changed_top1.sum()),
        reference_rescue=lib_report, runtime_s=round(time.monotonic()-started, 2),
        control_sha256=sha256(root/"submission_control.csv"),
        submission_sha256=sha256(root/"submission.csv"),
        baseline_source_sha256=pins["FROZEN_SHA256"],
        leaderboard_score=None, measured_improvement=None,
        default_policy_validated=bool(cfg.policy_file),
        warning="Full pretrained-ensemble validation requires fold-specific provenance; see review.")
    (root/"refactor_report.json").write_text(json.dumps(report, indent=2))
    report["output_certificate"] = final_certificate(root, test, "challenger")
    (root/"refactor_report.json").write_text(json.dumps(report, indent=2))
    return report
'''
(PACKAGE / 'pipeline.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/ranking.py
module_source = r'''"""Metric-aware union, masked channel fusion, bounded expansion and safe CSV output."""
from dataclasses import asdict, replace
from pathlib import Path
import json
import numpy as np
import pandas as pd
from .chemistry import identity, properties, unique_candidates
from .spectra import mass_consensus

FEATURES = ["neural_score", "neural_cos", "neural_fp", "neural_std", "neural_ok",
    "dual_score", "dual_cos", "dual_tanimoto", "smarts_consistency",
    "library_score", "library_entropy", "loss_entropy", "sqrt_cos", "matched", "library_coverage",
    "same_adduct", "same_instrument", "ce_delta", "ppm_error", "mass_reliability",
    "baseline_rank", "main_rank", "engine2_rank", "pubchem_rank", "prefusion_rank",
    "ice", "gl", "ice_present", "gl_present", "n_spectra", "n_adducts", "components", "charge"]


def build_union(evidence, rescued, queries, cfg):
    """Join by molecule AND metric identity; keep channel evidence without duplicate votes."""
    parts = [evidence.copy()]
    if len(rescued):
        parts.append(rescued.copy())
    f = pd.concat(parts, ignore_index=True)
    f["molecule_id"] = f.molecule_id.astype(str)
    f["metric_key"] = f.smiles.map(identity)
    f = f[f.metric_key.notna()].copy()
    rows = []
    for (mid, key), g in f.groupby(["molecule_id", "metric_key"], sort=False):
        rec = dict(molecule_id=mid, metric_key=key, smiles=g.smiles.iloc[0], processed=True)
        for col in f.columns.difference(["molecule_id", "smiles", "metric_key", "processed"]):
            v = pd.to_numeric(g[col], errors="coerce").dropna()
            if col.endswith("_rank"):
                v = v[v > 0]
                rec[col] = float(v.min()) if len(v) else 0.
            else:
                rec[col] = float(v.max()) if len(v) else np.nan
        rows.append(rec)
    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError("No valid candidate structures from any channel")
    query_map = {str(m): g for m, g in queries.groupby("molecule_id", sort=False)}
    parts = []
    for mid, g in out.groupby("molecule_id", sort=False):
        if mid not in query_map:
            raise ValueError(f"Candidate without query: {mid}")
        q = query_map[mid]
        mass, spread, hypotheses = mass_consensus(q, cfg.precursor_bias_ppm)
        g = g.copy()
        props = [properties(s) for s in g.smiles]
        g["components"] = [p["components"] for p in props]
        g["charge"] = [p["charge"] for p in props]
        cm = np.array([p["exact_mass"] for p in props])
        mh = np.array(hypotheses or ([mass] if np.isfinite(mass) else []))
        g["ppm_error"] = np.min(abs(cm[:, None]-mh[None, :])/np.maximum(mh, 1)*1e6, axis=1) if len(mh) else 1e5
        g["mass_reliability"] = float(np.isfinite(mass) and spread <= cfg.mass_ppm)
        g["n_spectra"] = len(q)
        g["n_adducts"] = q.adduct.nunique()
        # Mass mismatch is a soft feature: protomers/charged structures share metric identities.
        g["ice_present"] = g.get("ice", pd.Series(np.nan, index=g.index)).notna().astype(float)
        g["gl_present"] = g.get("gl", pd.Series(np.nan, index=g.index)).notna().astype(float)
        for col in ["baseline_rank", "main_rank", "engine2_rank", "pubchem_rank", "prefusion_rank", "library_rank"]:
            if col not in g:
                g[col] = 0.
        # Every baseline candidate survives pool truncation; reserve capacity for rescue rows.
        g["_priority"] = np.where(g.baseline_rank > 0, 1000-g.baseline_rank,
            1/np.maximum(1, g.main_rank.replace(0, 999))+
            1/np.maximum(1, g.engine2_rank.replace(0, 999))+
            g.get("library_score", pd.Series(0., index=g.index)).fillna(0))
        parts.append(g.sort_values(["_priority", "metric_key"], ascending=[False, True],
            kind="stable").head(cfg.pool_limit).drop(columns="_priority"))
    return pd.concat(parts, ignore_index=True)


def ranked_channel(values, valid):
    """Equal values share a rank; missing scores contribute no artificial negative vote."""
    values = np.asarray(values, dtype=float)
    valid = np.asarray(valid, dtype=bool) & np.isfinite(values)
    ranks = np.full(len(values), np.inf)
    ranks[valid] = pd.Series(-values[valid]).rank(method="min").to_numpy()
    return ranks


def heuristic_order(g, cfg):
    """Conservative RRF extension; its non-inherited weights are explicit hypotheses."""
    g = g.copy()
    n = len(g)
    score = np.zeros(n)
    base = g.baseline_rank.fillna(0).to_numpy()
    has_base = base > 0
    score[has_base] += 1/(cfg.rrf_offset+base[has_base])
    # Overflow receives one vote even when found by several strongly correlated engines.
    best = np.full(n, np.inf)
    for col in ["main_rank", "engine2_rank", "pubchem_rank", "prefusion_rank"]:
        r = g[col].fillna(0).to_numpy()
        best = np.minimum(best, np.where(r > 0, r, np.inf))
    score += cfg.overflow_weight/(cfg.rrf_offset+best)
    ok = g.get("neural_ok", pd.Series(False, index=g.index)).fillna(False).astype(bool).to_numpy()
    neural = g.get("neural_score", pd.Series(np.nan, index=g.index)).to_numpy()
    nr = ranked_channel(neural, ok)
    score += cfg.neural_weight/(cfg.rrf_offset+nr)
    lib = g.get("library_score", pd.Series(np.nan, index=g.index)).to_numpy()
    lr = ranked_channel(lib, np.isfinite(lib) & (lib > 0))
    # Strength matters: a poor match ranked #1 is weak evidence.
    score += cfg.library_weight*np.nan_to_num(lib, nan=0)**3/(cfg.rrf_offset+lr)
    dual = g.get("dual_score", pd.Series(np.nan, index=g.index)).to_numpy()
    dr = ranked_channel(dual, np.isfinite(dual))
    score += cfg.dual_weight/(cfg.rrf_offset+dr)
    return score


def apply_policy(cfg):
    """Load only a policy with recorded independent development selection provenance."""
    if not cfg.policy_file:
        return cfg
    p = json.loads(Path(cfg.policy_file).read_text())
    if p.get("selection_scope") != "nested_grouped_development":
        raise ValueError("Policy must identify nested grouped development provenance")
    allowed = {"neural_weight", "overflow_weight", "library_weight", "dual_weight", "max_new_in_top25"}
    unknown = set(p["parameters"])-allowed
    if unknown:
        raise ValueError(f"Unsupported policy parameters {unknown}")
    return replace(cfg, **p["parameters"]).validate()


def rank_candidates(frame, cfg, ranker=None):
    """Bounded candidate promotion; failed neural scores leave other channels available."""
    rows, audit = [], []
    for mid, g in frame.groupby("molecule_id", sort=False):
        g = g.copy().reset_index(drop=True)
        score = ranker.predict(g) if ranker is not None else heuristic_order(g, cfg)
        order = np.lexsort((g.metric_key.to_numpy(), -score))
        chosen, seen, nnew = [], set(), 0
        # A nearly exact, instrument/adduct-supported match can anchor rank 1. The threshold
        # is a policy hypothesis, not a calibrated confidence probability.
        anchored = []
        if "library_score" in g:
            strong = g[g.library_score.ge(cfg.protect_library_top1) & g.baseline_rank.eq(1)]
            if len(strong):
                anchored = [int(strong.index[0])]
        for i in anchored + order.tolist():
            if i in seen:
                continue
            new = not g.baseline_rank.iloc[i] > 0
            if new and nnew >= cfg.max_new_in_top25:
                continue
            seen.add(i); chosen.append(i); nnew += int(new)
            if len(chosen) == cfg.max_guesses:
                break
        out = g.iloc[chosen].copy()
        out["score"] = score[chosen]
        out["final_rank"] = np.arange(1, len(out)+1)
        rows.append(out)
        old = g[g.baseline_rank > 0].sort_values("baseline_rank").smiles.tolist()[:25]
        audit.append(dict(molecule_id=mid, candidates=len(g), output=len(out), new_promoted=nnew,
                          changed_top1=bool(old and old[0] != out.smiles.iloc[0])))
    return pd.concat(rows, ignore_index=True), pd.DataFrame(audit)


def validate_submission(frame, expected_ids):
    """Strict key coverage and valid unique chemistry, with no arbitrary filler molecules."""
    if frame.columns.tolist() != ["molecule_id", "smiles"] or frame.empty:
        raise ValueError("submission must have exactly molecule_id,smiles columns")
    if frame.isna().any().any() or not frame.molecule_id.is_unique:
        raise ValueError("Null or duplicate output row")
    if set(frame.molecule_id.astype(str)) != set(map(str, expected_ids)):
        raise ValueError("Output molecule IDs differ from actual test.parquet")
    for mid, text in frame.itertuples(index=False):
        guesses = text.split(";")
        keys = [identity(s) for s in guesses]
        if not 1 <= len(guesses) <= 25 or any(k is None for k in keys) or len(set(keys)) != len(keys):
            raise ValueError(f"Invalid/repeated/empty candidates for {mid}")
    return frame


def write_submission(ranked, test, path):
    """Validate before atomic replacement; sample_submission values are never copied."""
    ids = test.molecule_id.astype(str).drop_duplicates().tolist()
    lists = ranked.sort_values(["molecule_id", "final_rank"]).groupby("molecule_id").smiles.agg(";".join)
    out = pd.DataFrame({"molecule_id": ids, "smiles": [lists.get(mid, "") for mid in ids]})
    validate_submission(out, ids)
    target = Path(path); tmp = target.with_suffix(".csv.tmp")
    out.to_csv(tmp, index=False); tmp.replace(target)
    return out


class LearnedRanker:
    """Optional LightGBM ensemble; enforce the exact feature schema before scoring."""
    def __init__(self, directory):
        import lightgbm as lgb
        p = Path(directory)
        manifest = json.loads((p/"manifest.json").read_text())
        if manifest["features"] != FEATURES:
            raise ValueError("Ranker feature schema mismatch")
        self.models = [lgb.Booster(model_file=str(p/x)) for x in manifest["models"]]

    def predict(self, frame):
        x = frame.reindex(columns=FEATURES).astype(float).replace([np.inf, -np.inf], np.nan)
        return np.mean([m.predict(x, num_threads=4) for m in self.models], axis=0)
'''
(PACKAGE / 'ranking.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/runtime.py
module_source = r'''"""Bounded subprocess orchestration, file hashing and offline dependency preflight."""
from pathlib import Path
import hashlib
import os
import queue
import signal
import subprocess
import sys
import threading
import time


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8 << 20), b""):
            h.update(b)
    return h.hexdigest()


def run_logged(args, log, cwd, timeout, extra_env=None):
    """Stream output while enforcing a wall-clock timeout on the entire process group."""
    env = {**os.environ, "PYTHONHASHSEED": "0", "PYTHONUNBUFFERED": "1",
           "OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}
    env.pop("INC_PANEL", None)
    env.update(extra_env or {})
    proc = subprocess.Popen(list(map(str, args)), cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def read_output():
        for line in proc.stdout:
            messages.put(line)
        messages.put(None)
    threading.Thread(target=read_output, daemon=True).start()
    started = time.monotonic()
    try:
        done = False
        with open(log, "w") as handle:
            while not done:
                if time.monotonic()-started > timeout:
                    raise TimeoutError(f"Stage exceeded {timeout}s; see {log}")
                try:
                    line = messages.get(timeout=.5)
                except queue.Empty:
                    continue
                if line is None:
                    done = True
                else:
                    handle.write(line); handle.flush()
                    if len(line) < 5000:
                        print(line, end="", flush=True)
        status = proc.wait(timeout=10)
        if status:
            raise RuntimeError(f"Stage failed ({status}); see {log}")
    except BaseException:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait()
        raise


def bootstrap_rdkit(legacy, output):
    """Run the original wheel-only installer before the parent imports RDKit."""
    import json
    source = json.loads((Path(legacy)/"frozen_0465.ipynb").read_text())
    script = Path(output)/"bootstrap_metric.py"
    script.write_text("".join(source["cells"][2]["source"]))
    run_logged([sys.executable, script], Path(output)/"bootstrap.log", output, 180)
'''
(PACKAGE / 'runtime.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/scoring.py
module_source = r'''"""OOM-resilient scoring with the supplied checkpoint's exact featurizer and score equation."""
from pathlib import Path
import gc
import json
import sys
import numpy as np
import pandas as pd
import torch


@torch.inference_mode()
def score(bundle, checkpoint, evidence, queries, output, legacy, device="cuda"):
    """Preserve model semantics while adapting only the size of spectrum batches.

    Candidate encoding, peak filters, metadata and the neural score equation remain
    identical to the uploaded baseline. Missing/empty spectra receive neural_ok=False;
    RRF then omits the neural channel rather than treating its zero as a measured score.
    """
    sys.path.insert(0, str(legacy))
    from neural_scores import load_model
    torch.set_num_threads(4)
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Enable the T4 GPU for the supplied neural checkpoint")
    model, feat, cfg = load_model(bundle, checkpoint, device)
    frame = pd.read_parquet(evidence)
    query = pd.read_parquet(queries)
    qmap = {str(m): g for m, g in query.groupby("molecule_id", sort=False)}
    parts, errors, cache = [], [], {}
    for index, (mid, g) in enumerate(frame.groupby("molecule_id", sort=False)):
        g = g.sort_values("baseline_rank", kind="stable").copy()
        g["neural_ok"] = False
        for col in ["neural_cos", "neural_fp", "neural_score", "neural_std", "n_spectra", "n_adducts"]:
            g[col] = 0.
        if "processed" in g and not bool(g.processed.iloc[0]):
            parts.append(g); continue
        records = []
        for row in qmap[str(mid)].itertuples(index=False):
            try:
                pk, n = feat.peaks(row.ms2_mzs, row.ms2_normalized_intensities, row.precursor_mz, cfg["max_peaks"])
                if n < cfg["min_peaks"]:
                    continue
                m, a = feat.metadata(row.precursor_mz, row.adduct, row.ionization_mode, row.collision_energy_ev)
                records.append((pk, n, m, a))
            except (TypeError, ValueError) as exc:
                errors.append(dict(molecule_id=str(mid), stage="input", error=str(exc)))
        if not records:
            errors.append(dict(molecule_id=str(mid), stage="input", error="No usable spectra"))
            parts.append(g); continue
        fps = []
        for s in g.smiles:
            if s not in cache:
                cache[s] = feat.structure(s, cfg["fp_bits"])[-1]
            fps.append(cache[s])
        fp = torch.tensor(np.stack(fps), device=device, dtype=torch.float32)
        mol = model.encode_molecules(fp)
        cos_all, fp_all = [], []
        start, batch_size, failed = 0, 64, False
        while start < len(records):
            b = records[start:start+batch_size]
            try:
                x = torch.tensor(np.stack([v[0] for v in b]), device=device)
                lengths = torch.tensor([v[1] for v in b], device=device, dtype=torch.long)
                meta = torch.tensor(np.stack([v[2] for v in b]), device=device)
                adduct = torch.tensor([v[3] for v in b], device=device, dtype=torch.long)
                z, p = model.encode_spectra(x, lengths, meta, adduct)
                cos_all.append((z@mol.T).cpu().numpy())
                fp_all.append((p@fp.T/cfg["fp_bits"]).cpu().numpy())
                start += len(b)
            except torch.OutOfMemoryError:
                # Retrying changes batching only, never intensity transforms or candidate scores.
                gc.collect()
                if device == "cuda":
                    torch.cuda.empty_cache()
                if batch_size == 1:
                    failed = True
                    errors.append(dict(molecule_id=str(mid), stage="inference", error="OOM at batch_size=1"))
                    break
                batch_size = max(1, batch_size//2)
        if not failed:
            cos, fpscore = np.concatenate(cos_all), np.concatenate(fp_all)
            combined = cos/cfg["temperature"]+cfg["fp_score_weight"]*fpscore
            if not np.isfinite(combined).all():
                errors.append(dict(molecule_id=str(mid), stage="inference", error="Nonfinite model scores"))
            else:
                g["neural_cos"] = cos.mean(0); g["neural_fp"] = fpscore.mean(0)
                g["neural_score"] = combined.mean(0); g["neural_std"] = combined.std(0)
                g["n_spectra"] = len(records); g["n_adducts"] = qmap[str(mid)].adduct.nunique()
                g["neural_ok"] = True
        parts.append(g)
        if (index+1)%25 == 0:
            print("NEURAL SCORED", index+1, "molecules", flush=True)
    result = pd.concat(parts, ignore_index=True)
    if len(result) != len(frame):
        raise AssertionError("Scoring lost candidate rows")
    result.to_parquet(output, index=False)
    Path(str(output)+".errors.json").write_text(json.dumps(errors, indent=2))
    print("NEURAL FEATURES", result.shape, "issues", len(errors), flush=True)


if __name__ == "__main__":
    score(*sys.argv[1:])
'''
(PACKAGE / 'scoring.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/spectra.py
module_source = r'''"""Robust spectrum cleaning, exact adduct arithmetic and multiscale mass tokens."""
from dataclasses import dataclass
import numpy as np

PROTON = 1.007276466621
ELECTRON = 0.000548579909
WATER = 18.01056468403
ADDUCTS = {"[M+H]+": PROTON, "[M+NH4]+": 18.033825553,
    "[M-H2O+H]+": PROTON-WATER, "[M-2H2O+H]+": PROTON-2*WATER,
    "[M+Na]+": 22.9897692820-ELECTRON, "[M+K]+": 38.9637064864-ELECTRON,
    "[M-H]-": -PROTON, "[M-H2O-H]-": -PROTON-WATER,
    "[M+CH2O2-H]-": 46.0054793036-PROTON, "[M+Cl]-": 34.968852682+ELECTRON}
ADDUCT_VOCAB = {a: i+1 for i, a in enumerate(ADDUCTS)}
COMMON_LOSSES = np.array([17.0265491, 18.0105647, 27.9949146, 28.0313001, 30.0105647,
    43.9898292, 46.0054793, 60.0211294, 79.9663309, 97.9768956,
    120.0422587, 132.0422587, 146.0579088, 162.0528234, 176.0320880], dtype=float)


def finite_float(x, default=np.nan):
    try:
        v = float(x)
        return v if np.isfinite(v) else default
    except (TypeError, ValueError):
        return default


def energies(value):
    """Preserve merged eV acquisitions; never reinterpret unresolved NCE as eV."""
    try:
        a = np.atleast_1d(np.asarray(value, dtype=float)).ravel()
        return a[np.isfinite(a) & (a >= 0)]
    except (TypeError, ValueError):
        return np.empty(0)


def neutral_hypotheses(precursor, adduct, mode, bias_ppm=0.):
    """Exact mass inversion; unknown adducts retain multiple same-polarity hypotheses."""
    p = finite_float(precursor) / (1 + bias_ppm*1e-6)
    if not np.isfinite(p) or p <= 0:
        return []
    if adduct in ADDUCTS:
        return [(p-ADDUCTS[adduct], 1.)]
    keys = ["[M-H]-", "[M+CH2O2-H]-", "[M+Cl]-"] if mode == "negative" else [
        "[M+H]+", "[M+Na]+", "[M+NH4]+", "[M-H2O+H]+", "[M+K]+"]
    return [(p-ADDUCTS[a], 1/len(keys)) for a in keys if p > ADDUCTS[a]]


def mass_consensus(frame, bias_ppm=0.):
    """Robust consensus of *reported* adduct masses, plus alternate hypotheses and MAD.

    This estimates relative agreement, not an absolute instrument calibration. A global
    correction is applied only when supplied from independent lock-mass measurements.
    """
    known, alternatives = [], []
    for r in frame.itertuples(index=False):
        h = neutral_hypotheses(r.precursor_mz, r.adduct, r.ionization_mode, bias_ppm)
        alternatives.extend(m for m, _ in h)
        if r.adduct in ADDUCTS and h:
            known.append(h[0][0])
    if known:
        a = np.array(known)
        center = float(np.median(a))
        mad_ppm = float(np.median(np.abs(a-center))/center*1e6)
        return center, mad_ppm, sorted(set(known + alternatives))
    return np.nan, np.nan, sorted(set(alternatives))


@dataclass
class Spectrum:
    mz: np.ndarray
    intensity: np.ndarray
    precursor: float
    adduct: str
    mode: str
    ce: np.ndarray
    instrument: str
    raw_base: float

    @property
    def quality(self):
        """Bounded quality weight; unknown raw counts are neutral, not zero quality."""
        if not len(self.mz):
            return 0.
        count = min(1., np.sqrt(len(self.mz)/20.))
        raw = .8 if not np.isfinite(self.raw_base) else np.clip(np.log10(max(self.raw_base, 1))/5, .2, 1.)
        return float(count*raw)


def clean_arrays(mz, intensity, precursor, cfg):
    """Finite/aligned peaks, base-peak relative floor, duplicates and local top-N.

    Precursor ±0.02 Da and isotope-region peaks are excluded from structural matching.
    Low-peak spectra survive; no invented peak or zero-length Transformer sequence.
    Deisotoping is deliberately conservative: close real fragments are never assumed C13.
    """
    try:
        m = np.asarray(mz, dtype=float).ravel()
        y = np.asarray(intensity, dtype=float).ravel()
    except (ValueError, TypeError):
        return np.empty(0), np.empty(0)
    if len(m) != len(y):
        return np.empty(0), np.empty(0)
    valid = np.isfinite(m) & np.isfinite(y) & (m > 0) & (y > 0)
    p = finite_float(precursor)
    if np.isfinite(p):
        valid &= m < p-.02
    m, y = m[valid], y[valid]
    if not len(m):
        return m, y
    y /= y.max()
    keep = y >= cfg.floor
    m, y = m[keep], y[keep]
    order = np.argsort(m, kind="stable")
    m, y = m[order], y[order]
    # Merge exact duplicate centroids only; preserve genuine nearby peaks.
    u, inv = np.unique(m, return_inverse=True)
    v = np.zeros(len(u)); np.add.at(v, inv, y)
    m, y = u, v/v.max()
    windows = np.floor(m/cfg.window_da).astype(int)
    chosen = []
    for w in np.unique(windows):
        ids = np.flatnonzero(windows == w)
        chosen.extend(ids[np.argsort(-y[ids], kind="stable")[:cfg.window_peaks]])
    chosen = np.asarray(chosen, dtype=int)
    if len(chosen) > cfg.max_peaks:
        chosen = chosen[np.argsort(-y[chosen], kind="stable")[:cfg.max_peaks]]
    chosen.sort()
    m, y = m[chosen], y[chosen]
    return m, y/y.sum()


def from_row(row, cfg):
    m, y = clean_arrays(row.ms2_mzs, row.ms2_normalized_intensities, row.precursor_mz, cfg)
    return Spectrum(m, y, finite_float(row.precursor_mz), str(row.adduct), str(row.ionization_mode),
                    energies(getattr(row, "collision_energy_ev", None)),
                    str(getattr(row, "instrument_type", "unknown")),
                    finite_float(getattr(row, "base_peak_intensity", None)))


def entropy_weights(y):
    """Entropy-dependent intensity weighting from Li et al. (2021)."""
    if not len(y):
        return y
    y = y/y.sum()
    h = -np.sum(y*np.log(np.maximum(y, 1e-30)))
    z = y**(.25+.25*h) if h < 3 else y.copy()
    return z/z.sum()


def aligned_scores(am, ay, bm, by, cfg):
    """One-to-one maximum-product greedy alignment; entropy, cosine, matched fraction.

    Edges are considered by strength, preventing a crowded reference region from matching
    one query peak repeatedly. For exact accelerated library search use ms_entropy; this
    bounded candidate reranker is deliberately NumPy-only and handles a ppm + Da tolerance.
    """
    if not len(am) or not len(bm):
        return 0., 0., 0.
    ap, bp = entropy_weights(ay), entropy_weights(by)
    edges = []
    for i, mass in enumerate(am):
        tol = max(cfg.fragment_da, cfg.fragment_ppm*1e-6*mass)
        lo, hi = np.searchsorted(bm, [mass-tol, mass+tol])
        edges.extend((float(ay[i]*by[j]), i, j) for j in range(lo, hi))
    edges.sort(reverse=True)
    ai, bi, pair = set(), set(), []
    for _, i, j in edges:
        if i not in ai and j not in bi:
            ai.add(i); bi.add(j); pair.append((i, j))
    if not pair:
        return 0., 0., 0.
    a, b = np.array(pair).T
    f = lambda x: x*np.log2(np.maximum(x, 1e-30))
    ent = .5*np.sum(f(ap[a]+bp[b])-f(ap[a])-f(bp[b]))
    cos = np.sum(np.sqrt(ay[a]*by[b]))  # cosine of L2-normalized sqrt intensities
    matched = .5*(np.sum(ay[a])+np.sum(by[b]))
    return float(np.clip(ent, 0, 1)), float(np.clip(cos, 0, 1)), float(matched)


def compare(query, ref, cfg):
    """Fragments and neutral losses form separate, correlated evidence features."""
    direct = aligned_scores(query.mz, query.intensity, ref.mz, ref.intensity, cfg)
    qm, rm = query.precursor-query.mz, ref.precursor-ref.mz
    qo, ro = np.argsort(qm), np.argsort(rm)
    loss = aligned_scores(qm[qo], query.intensity[qo], rm[ro], ref.intensity[ro], cfg)
    ce_delta = abs(query.ce.mean()-ref.ce.mean()) if len(query.ce) and len(ref.ce) else 30.
    same_mode = float(query.mode == ref.mode)
    same_adduct = float(query.adduct == ref.adduct)
    same_inst = float("tims" in query.instrument.lower() and "tims" in ref.instrument.lower())
    # Cross-mode evidence is retained as a feature; it is strongly attenuated in heuristic search.
    score = max(direct[0], .85*loss[0])*(.25+.75*same_mode)*(.8+.2*same_adduct)
    return dict(entropy=direct[0], sqrt_cos=direct[1], matched=direct[2],
                loss_entropy=loss[0], library_score=score, same_mode=same_mode,
                same_adduct=same_adduct, same_instrument=same_inst, ce_delta=ce_delta)


def peak_tokens(spec, limit=96):
    """Continuous Fourier-ready mass/loss tokens and explicit mass-defect channels."""
    ids = np.argsort(-spec.intensity, kind="stable")[:limit]
    ids = ids[np.argsort(spec.mz[ids])]
    m, y = spec.mz[ids], spec.intensity[ids]
    loss = spec.precursor-m
    tokens = np.column_stack([m/1200, loss/1200, np.sqrt(y), np.log1p(1000*y)/7,
                              m-np.round(m), loss-np.round(loss)]).astype(np.float32)
    return tokens


def difference_tokens(spec, limit=32, top_peaks=32):
    """Top pairwise fragment mass differences, weighted without O(N²) on full spectra."""
    ids = np.argsort(-spec.intensity, kind="stable")[:top_peaks]
    m, y = spec.mz[ids], spec.intensity[ids]
    i, j = np.triu_indices(len(ids), 1)
    d, w = np.abs(m[i]-m[j]), np.sqrt(y[i]*y[j])
    keep = (d > .05) & (d < 1200)
    d, w = d[keep], w[keep]
    order = np.argsort(-w, kind="stable")[:limit]
    d, w = d[order], w[order]
    return np.column_stack([d/1200, np.zeros(len(d)), w,
        np.log1p(1000*w)/7, d-np.round(d), np.zeros(len(d))]).astype(np.float32)


def metadata(spec):
    """Condition on adduct, polarity, platform, CE mean/range/count and missingness."""
    h = neutral_hypotheses(spec.precursor, spec.adduct, spec.mode)
    ce = spec.ce
    return np.array([spec.precursor/1200, (h[0][0] if h else 0)/1200,
        float(spec.mode == "positive"), float(spec.mode == "negative"),
        float("tims" in spec.instrument.lower()), ce.mean()/100 if len(ce) else 0,
        np.ptp(ce)/100 if len(ce) else 0, min(len(ce), 8)/8,
        float(not len(ce)), len(spec.mz)/160, spec.quality], dtype=np.float32)
'''
(PACKAGE / 'spectra.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/training.py
module_source = r'''"""Trainable spectrum/structure dual encoder. Training is a separate preparation job.

No randomly initialized model is used during submission. Existing checkpoints retain
their original preprocessing. This module's checkpoint and fingerprint schema are new.
"""
from dataclasses import dataclass, asdict
from pathlib import Path
import json
import math
import random
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from .chemistry import fingerprint, FP_BITS, FP_SCHEMA, properties
from .spectra import from_row, peak_tokens, difference_tokens, metadata, ADDUCT_VOCAB
from .config import Config


@dataclass
class ModelConfig:
    dim: int = 192
    layers: int = 4
    heads: int = 6
    dropout: float = .15
    tokens: int = 128
    fp_bits: int = FP_BITS
    fp_schema: str = FP_SCHEMA
    temperature: float = .08


class DualEncoder(nn.Module):
    """Permutation-invariant mass-token Transformer with fingerprint and retrieval heads.

    Fourier masses retain fine mass defects. Token types distinguish measured peaks from
    pairwise differences; differences never pretend to be fragment ions. A CLS token
    remains unmasked even for completely empty input spectra.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("frequencies", torch.tensor([1., 4., 16., 64., 256., 1024., 4096., 16384.]))
        self.project = nn.Linear(38, cfg.dim)  # 6 + 2 masses × 8 frequencies × sin/cos
        self.types = nn.Embedding(3, cfg.dim)
        self.adducts = nn.Embedding(11, cfg.dim)
        self.meta = nn.Sequential(nn.Linear(11, cfg.dim), nn.GELU(), nn.Linear(cfg.dim, cfg.dim))
        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.dim))
        layer = nn.TransformerEncoderLayer(cfg.dim, cfg.heads, cfg.dim*4, cfg.dropout,
            batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, cfg.layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(cfg.dim)
        self.fp = nn.Linear(cfg.dim, cfg.fp_bits)
        self.molecule = nn.Sequential(nn.Linear(cfg.fp_bits, 512), nn.GELU(), nn.Dropout(cfg.dropout),
                                      nn.Linear(512, cfg.dim), nn.LayerNorm(cfg.dim))

    def encode_spectra(self, tokens, types, valid, meta, adduct):
        phase = 2*math.pi*tokens[..., :2, None]*self.frequencies
        mass = torch.cat([phase.sin(), phase.cos()], dim=-1).flatten(-2)
        x = self.project(torch.cat([tokens, mass], dim=-1))+self.types(types)
        condition = self.meta(meta)+self.adducts(adduct)
        cls = self.cls.expand(len(x), 1, -1)+condition[:, None]
        x = torch.cat([cls, x+condition[:, None]], dim=1)
        pad = torch.cat([torch.zeros(len(x), 1, device=x.device, dtype=torch.bool), ~valid], dim=1)
        z = self.norm(self.encoder(x, src_key_padding_mask=pad)[:, 0])
        return F.normalize(z, dim=-1), self.fp(z)

    def encode_molecules(self, fps):
        return F.normalize(self.molecule(fps), dim=-1)


def tokenize(spec, cfg):
    """Fixed-size float16-compatible tokens and an explicit nonpadding mask."""
    peaks = peak_tokens(spec, min(96, cfg.tokens))
    diffs = difference_tokens(spec, max(0, cfg.tokens-len(peaks)))
    x = np.zeros((cfg.tokens, 6), np.float32)
    types = np.zeros(cfg.tokens, np.int64)
    valid = np.zeros(cfg.tokens, bool)
    v = np.concatenate([peaks, diffs])[:cfg.tokens]
    x[:len(v)] = v; valid[:len(v)] = True
    types[:len(peaks)] = 1; types[len(peaks):len(v)] = 2
    return x, types, valid, metadata(spec), ADDUCT_VOCAB.get(spec.adduct, 0)


def prepare_store(train_path, split_manifest, destination, cfg=Config(), model_cfg=ModelConfig()):
    """One Arrow pass to disk-backed token arrays; fingerprints stored once per skeleton.

    The split manifest is created BEFORE augmentation. No spectra from an excluded key
    or scaffold may enter another fold through a different adduct/source library.
    """
    dest = Path(destination); dest.mkdir(parents=True, exist_ok=True)
    if (dest/"manifest.json").exists():
        raise FileExistsError("Use a new feature-store directory")
    structures = split_manifest.reset_index(drop=True).copy()
    if not structures.metric_key.is_unique:
        raise ValueError("Split manifest needs one row per metric identity")
    structures["exact_mass"] = [properties(s)["exact_mass"] for s in structures.smiles]
    structures.to_parquet(dest/"structures.parquet", index=False)
    key_index = {str(k): i for i, k in enumerate(structures.metric_key)}
    fp = np.lib.format.open_memmap(dest/"fingerprints.npy", mode="w+", dtype="uint8", shape=(len(structures), FP_BITS))
    for i, s in enumerate(structures.smiles):
        fp[i] = fingerprint(s)
    fp.flush()
    pf = pq.ParquetFile(train_path); capacity = pf.metadata.num_rows
    specs = {
        "tokens": ("float16", (capacity, model_cfg.tokens, 6)),
        "types": ("uint8", (capacity, model_cfg.tokens)),
        "valid": ("bool", (capacity, model_cfg.tokens)),
        "meta": ("float32", (capacity, 11)),
        "adduct": ("uint8", (capacity,)), "structure": ("int32", (capacity,)),
    }
    arrays = {k: np.lib.format.open_memmap(dest/(k+".npy"), mode="w+", dtype=d, shape=sh)
              for k, (d, sh) in specs.items()}
    n = 0
    from .chemistry import identity
    columns = [c for c in ["inchikey14", "normalized_smiles", "ms2_mzs", "ms2_normalized_intensities",
        "precursor_mz", "adduct", "ionization_mode", "collision_energy_ev", "instrument_type",
        "base_peak_intensity", "precursor_error_ppm"] if c in pf.schema_arrow.names]
    for batch in pf.iter_batches(batch_size=8192, columns=columns):
        for row in batch.to_pandas().itertuples(index=False):
            err = getattr(row, "precursor_error_ppm", np.nan)
            if err is not None and np.isfinite(err) and abs(err) > cfg.fallback_ppm:
                continue
            # Use canonical keys, never trust a raw-key alias crossing a tautomer boundary.
            key = identity(row.normalized_smiles)
            if key not in key_index:
                continue
            x, t, valid, meta, ad = tokenize(from_row(row, cfg), model_cfg)
            if not valid.any():
                continue
            for k, value in zip(["tokens", "types", "valid", "meta", "adduct", "structure"],
                                    [x, t, valid, meta, ad, key_index[key]]):
                arrays[k][n] = value
            n += 1
    for a in arrays.values():
        a.flush()
    record = dict(rows=n, capacity=capacity, structures=len(structures),
                  model=asdict(model_cfg), preprocessing=asdict(cfg), schema=FP_SCHEMA)
    (dest/"manifest.json").write_text(json.dumps(record, indent=2))
    return record


class StoreDataset(torch.utils.data.Dataset):
    """A small row-index view over memory maps, safe with zero worker processes on Kaggle."""
    def __init__(self, store, rows):
        self.root = Path(store); self.rows = np.asarray(rows)
        self.arrays = {k: np.load(self.root/(k+".npy"), mmap_mode="r") for k in
                       ["tokens", "types", "valid", "meta", "adduct", "structure", "fingerprints"]}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        r = int(self.rows[index]); sid = int(self.arrays["structure"][r])
        d = {k: np.array(self.arrays[k][r], copy=True) for k in ["tokens", "types", "valid", "meta", "adduct"]}
        return d, np.array(self.arrays["fingerprints"][sid], dtype=np.float32), sid


def tensor_batch(d, device):
    return [d["tokens"].to(device).float(), d["types"].to(device).long(), d["valid"].to(device).bool(),
            d["meta"].to(device).float(), d["adduct"].to(device).long()]


def contrastive_loss(z, mol, keys, temperature):
    """Symmetric multi-positive InfoNCE; identical molecules never become false negatives."""
    logits = z@mol.T/temperature
    positive = keys[:, None].eq(keys[None, :])
    def side(a, mask):
        return -(torch.logsumexp(a.masked_fill(~mask, -torch.inf), 1)-torch.logsumexp(a, 1)).mean()
    return .5*(side(logits, positive)+side(logits.T, positive.T))


@torch.inference_mode()
def validate_retrieval(model, dataset, structures, fps, device, max_queries=512):
    """Development MRR@25 against the unmodified full mass-gated structure catalogue.

    Candidate recall and rank are separate; no forced truth insertion. All development
    spectra for a selected structure are averaged, capped at four to bound epoch cost.
    """
    rows_by_key = {}
    for i, r in enumerate(dataset.rows):
        sid = int(dataset.arrays["structure"][r])
        if sid not in rows_by_key and len(rows_by_key) >= max_queries:
            continue
        if len(rows_by_key.setdefault(sid, [])) < 4:
            rows_by_key[sid].append(i)
    model.eval(); masses = structures.exact_mass.to_numpy(); rr = []; recalls = []
    for sid, indices in rows_by_key.items():
        # Measured mass from encoded metadata, not the known answer formula/mass.
        observed = [float(dataset[i][0]["meta"][1])*1200 for i in indices]
        target = float(np.median(observed))
        cand = np.flatnonzero(abs(masses-target) <= max(.003, target*35e-6))
        recalls.append(int(sid in cand))
        if sid not in cand:
            rr.append(0.); continue
        batch = torch.utils.data.default_collate([dataset[i] for i in indices])
        z, logits = model.encode_spectra(*tensor_batch(batch[0], device))
        fp = torch.tensor(np.asarray(fps[cand]), device=device, dtype=torch.float32)
        mol = model.encode_molecules(fp)
        pred = logits.sigmoid().mean(0)
        inter = pred@fp.T
        tanimoto = inter/(pred.sum()+fp.sum(1)-inter).clamp_min(1e-6)
        score = (z.mean(0)@mol.T+ .2*tanimoto).cpu().numpy()
        order = np.lexsort((cand, -score))
        rank = int(np.flatnonzero(cand[order] == sid)[0])+1
        rr.append(1/rank if rank <= 25 else 0.)
    return dict(mrr25=float(np.mean(rr)) if rr else 0., recall=float(np.mean(recalls)) if rr else 0.,
                queries=len(rr))


def fit_dual(store, output, train_folds, dev_folds, test_folds, epochs=15, batch_size=128, seed=2026):
    """Fit on training folds and early-stop on development retrieval MRR; never test-tune."""
    train_folds, dev_folds, test_folds = map(set, [train_folds, dev_folds, test_folds])
    if not train_folds or not dev_folds or not test_folds or any([
        train_folds & dev_folds, train_folds & test_folds, dev_folds & test_folds]):
        raise ValueError("Supply nonempty disjoint train/development/outer-test folds")
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.set_num_threads(4)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    root, dest = Path(store), Path(output); dest.mkdir(parents=True, exist_ok=True)
    if (dest/"model.pt").exists():
        raise FileExistsError("Use a fresh checkpoint directory")
    manifest = json.loads((root/"manifest.json").read_text()); n = manifest["rows"]
    structures = pd.read_parquet(root/"structures.parquet")
    for col in ["metric_key", "group"]:
        sets = [set(structures.loc[structures.fold.isin(f), col]) for f in [train_folds, dev_folds, test_folds]]
        if any(sets[i] & sets[j] for i, j in [(0,1), (0,2), (1,2)]):
            raise ValueError(f"Cross-fold {col} leakage")
    sid = np.load(root/"structure.npy", mmap_mode="r")[:n]
    rowfold = structures.fold.to_numpy()[sid]
    tr = np.flatnonzero(np.isin(rowfold, list(train_folds)))
    dv = np.flatnonzero(np.isin(rowfold, list(dev_folds)))
    if not len(tr) or not len(dv):
        raise ValueError("Empty training/development feature store")
    devset = StoreDataset(root, dv)
    model_cfg = ModelConfig(**manifest["model"]); model = DualEncoder(model_cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=.02)
    scaler = torch.amp.GradScaler("cuda", enabled=device=="cuda")
    fps = np.load(root/"fingerprints.npy", mmap_mode="r")
    # Bit frequency from TRAINING STRUCTURES only, never validation or catalogue statistics.
    freq = np.asarray(fps[np.unique(sid[tr])], dtype=np.float32).mean(0)
    pos = torch.tensor(np.clip((1-freq)/np.maximum(freq, .01), 1, 10), device=device)
    order = np.argsort(sid[tr], kind="stable"); grouped = np.split(tr[order], np.flatnonzero(np.diff(sid[tr][order]))+1)
    history = []; best = -1.; stale = 0
    for epoch in range(epochs):
        rng = np.random.default_rng(seed+epoch)
        matched = np.load(root/"meta.npy", mmap_mode="r")
        selected = []
        for r in grouped:
            tims = r[matched[r, 4] > .5]
            pool = tims if len(tims) and rng.random()<.65 else r
            selected.append(int(rng.choice(pool)))  # uniform structures; platform-aware view
        # Alternate global negatives and mass-neighbor hard negatives. Same-identity
        # examples remain positives in InfoNCE even if a custom sampler repeats them.
        hard_negative_epoch = bool(epoch%2)
        if hard_negative_epoch:
            selected = np.array(selected)
            mass = structures.exact_mass.to_numpy()[sid[selected]]
            selected = selected[np.argsort(mass, kind="stable")]
            chunks = [selected[i:i+batch_size] for i in range(0,len(selected),batch_size)]
            rng.shuffle(chunks); selected = np.concatenate(chunks)
        ds = StoreDataset(root, selected)
        loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=not hard_negative_epoch,
            num_workers=0, pin_memory=device=="cuda", generator=torch.Generator().manual_seed(seed+epoch))
        model.train(); loss_sum = 0.
        for d, target, keys in loader:
            target, keys = target.to(device), keys.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=device=="cuda", dtype=torch.float16):
                z, fp_logits = model.encode_spectra(*tensor_batch(d, device))
                mz = model.encode_molecules(target)
                bce = F.binary_cross_entropy_with_logits(fp_logits, target, pos_weight=pos)
                loss = bce+.25*contrastive_loss(z, mz, keys, model_cfg.temperature)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            scaler.step(optimizer); scaler.update()
            loss_sum += float(loss.detach())*len(target)
        metric = validate_retrieval(model, devset, structures, fps, device)
        history.append(dict(epoch=epoch+1, loss=loss_sum/len(ds), **metric))
        print(history[-1], flush=True)
        if metric["mrr25"] > best:
            best = metric["mrr25"]; stale = 0
            torch.save(dict(state_dict=model.cpu().state_dict(), model=asdict(model_cfg),
                preprocessing=manifest["preprocessing"], fp_schema=FP_SCHEMA,
                train_folds=sorted(train_folds), dev_folds=sorted(dev_folds), test_folds=sorted(test_folds),
                trained_metric_keys=structures.loc[structures.fold.isin(train_folds), "metric_key"].tolist(),
                trained_groups=structures.loc[structures.fold.isin(train_folds), "group"].tolist()), dest/"model.pt")
            model.to(device)
        else:
            stale += 1
        if stale >= 4:
            break
    (dest/"history.json").write_text(json.dumps(history, indent=2))
    return history


@torch.inference_mode()
def predict_candidates(checkpoint, queries, candidates, cfg, device=None):
    """Optional trained model scores; masks failed spectra and computes soft Tanimoto."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if ck["fp_schema"] != FP_SCHEMA:
        raise ValueError("Fingerprint target schema mismatch")
    mc = ModelConfig(**ck["model"]); model = DualEncoder(mc).to(device)
    model.load_state_dict(ck["state_dict"]); model.eval()
    preprocessing = Config(**ck["preprocessing"])
    qmap = {str(m): g for m, g in queries.groupby("molecule_id", sort=False)}
    parts = []
    for mid, g in candidates.groupby("molecule_id", sort=False):
        specs = [from_row(r, preprocessing) for r in qmap[str(mid)].itertuples(index=False)]
        records = [tokenize(s, mc) for s in specs]
        x, ty, va, me, ad = [torch.tensor(np.stack([r[i] for r in records]), device=device) for i in range(5)]
        z, logits = model.encode_spectra(x.float(), ty.long(), va.bool(), me.float(), ad.long())
        weights = torch.tensor([s.quality for s in specs], device=device)
        if not weights.sum() > 0:
            g = g.copy(); g["dual_score"] = np.nan; parts.append(g); continue
        weights /= weights.sum()
        pred = weights@logits.sigmoid()
        fp = torch.tensor(np.stack([fingerprint(s) for s in g.smiles]), device=device, dtype=torch.float32)
        mol = model.encode_molecules(fp)
        inter = pred@fp.T
        tanimoto = inter/(pred.sum()+fp.sum(1)-inter).clamp_min(1e-6)
        g = g.copy()
        g["dual_cos"] = ((weights@z)@mol.T).cpu().numpy()
        g["dual_tanimoto"] = tanimoto.cpu().numpy()
        # Explicit SMARTS consistency is a likelihood feature, never an absolute veto.
        p = pred[-32:].clamp(1e-5, 1-1e-5); bits = fp[:, -32:]
        g["smarts_consistency"] = (bits*p.log()+(1-bits)*(1-p).log()).mean(1).cpu().numpy()
        g["dual_score"] = g.dual_cos+.2*g.dual_tanimoto
        parts.append(g)
    return pd.concat(parts, ignore_index=True)


def mass_gated_vector_search(query_embedding, embeddings, masses, target_mass, ppm=35., k=192):
    """Exact mass gate BEFORE vector retrieval. Optional FAISS inner-product acceleration.

    Binary FAISS Hamming distance is not Tanimoto; this helper indexes continuous learned
    embeddings only. Candidate fingerprint agreement is computed exactly after retrieval.
    """
    ids = np.flatnonzero(abs(np.asarray(masses)-target_mass) <= target_mass*ppm*1e-6)
    if not len(ids):
        return ids
    x = np.ascontiguousarray(embeddings[ids], dtype=np.float32)
    q = np.ascontiguousarray(np.asarray(query_embedding, dtype=np.float32).reshape(1, -1))
    try:
        import faiss
        index = faiss.IndexFlatIP(x.shape[1]); index.add(x)
        _, selected = index.search(q, min(k, len(ids)))
        return ids[selected[0]]
    except ImportError:
        score = x@q[0]
        return ids[np.lexsort((ids, -score))[:k]]
'''
(PACKAGE / 'training.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/validation.py
module_source = r'''"""Canonical scaffold grouping, honest MRR denominators and nested ranker validation."""
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from .chemistry import identity, group_key
from .ranking import FEATURES


def split_manifest(structures, folds=5, seed=2026):
    """Union scaffold and tautomer-equivalence constraints BEFORE balancing folds.

    The same metric key can have different drawn Murcko scaffolds. Connected components
    prevent either identity or scaffold leakage. Acyclic compounds form one conservative
    group; report its size rather than silently random-splitting it.
    """
    if folds < 3:
        raise ValueError("Use at least three folds for train/development/outer test")
    frame = structures.copy()
    if "smiles" not in frame:
        frame = frame.rename(columns={"normalized_smiles": "smiles"})
    frame["metric_key"] = frame.smiles.map(identity)
    if frame.metric_key.isna().any():
        raise ValueError("Invalid labelled structures must be reviewed before splitting")
    frame["scaffold"] = frame.smiles.map(group_key)
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[max(a,b)] = min(a,b)
    for r in frame.itertuples():
        union("K:"+r.metric_key, "S:"+r.scaffold)
    frame["group"] = [find("K:"+k) for k in frame.metric_key]
    group_sizes = frame.groupby("group").metric_key.nunique()
    if len(group_sizes) < folds:
        raise ValueError("Fewer scaffold components than requested folds")
    stable = lambda g: hashlib.sha256(f"{seed}:{g}".encode()).hexdigest()
    groups = sorted(group_sizes.index, key=lambda g: (-group_sizes[g], stable(g)))
    loads, assignment = np.zeros(folds, int), {}
    for group in groups:
        fold = int(np.argmin(loads)); assignment[group] = fold; loads[fold] += group_sizes[group]
    frame["fold"] = frame.group.map(assignment)
    result = frame.drop_duplicates("metric_key").reset_index(drop=True)
    for i in range(folds):
        a, b = frame[frame.fold == i], frame[frame.fold != i]
        if set(a.metric_key)&set(b.metric_key) or set(a.scaffold)&set(b.scaffold):
            raise AssertionError("Split leakage")
    result.attrs["fold_sizes"] = loads.tolist()
    result.attrs["largest_component"] = int(group_sizes.max())
    return result


def mrr25(predictions, labels):
    """All labelled queries are in the denominator, including missing/candidate-miss rows."""
    if not labels.molecule_id.is_unique:
        raise ValueError("Labels require one metric identity per molecule")
    pred = {str(m): g.sort_values("final_rank").metric_key.tolist()[:25]
            for m, g in predictions.groupby("molecule_id", sort=False)}
    rr, hit1, hit25 = [], [], []
    for row in labels.itertuples(index=False):
        candidates = pred.get(str(row.molecule_id), [])
        rank = candidates.index(row.metric_key)+1 if row.metric_key in candidates else 0
        rr.append(1/rank if rank else 0.)
        hit1.append(int(rank==1)); hit25.append(int(rank>0))
    return dict(mrr25=float(np.mean(rr)) if rr else 0.,
        recall25=float(np.mean(hit25)) if rr else 0., top1=float(np.mean(hit1)) if rr else 0.,
        queries=len(rr))


def audit_feature_provenance(provenance, labels, split, required_predictors=("dual",)):
    """Fail closed if held-out features used a predictor trained on those structures.

    Spectral references and model-training keys are separate provenance: class-1 library
    identification may use another acquisition of that structure, but model fitting still
    excludes the held-out chemical group. Class-2 replay excludes all query-group references.
    """
    required = {"predictor_id", "outer_fold", "feature_fold", "trained_metric_keys", "trained_groups", "query_metric_keys",
                "query_groups", "reference_excluded_groups", "regime"}
    for record in provenance:
        if required-set(record):
            raise ValueError(f"Missing provenance fields {required-set(record)}")
        fold = record["feature_fold"]
        outer = record["outer_fold"]
        folds = sorted(split.fold.unique())
        dev = folds[(folds.index(outer)+1)%len(folds)]
        expected = set(split.loc[split.fold.eq(fold), "metric_key"])
        keys, groups = set(record["query_metric_keys"]), set(record["query_groups"])
        if not keys <= expected or keys & set(record["trained_metric_keys"]):
            raise ValueError("Feature predictor overlaps held-out keys")
        if groups & set(record["trained_groups"]):
            raise ValueError("Feature predictor overlaps held-out scaffold groups")
        forbidden = split[split.fold.isin([outer, dev])]
        if set(forbidden.metric_key) & set(record["trained_metric_keys"]) or set(forbidden.group) & set(record["trained_groups"]):
            raise ValueError("Nested base features used outer-test/development labels")
        if record["regime"] == "class2" and not groups <= set(record["reference_excluded_groups"]):
            raise ValueError("Class-2 reference index contains held-out groups")
    for predictor in required_predictors:
        records = [r for r in provenance if r["predictor_id"]==predictor]
        covered = set().union(*(set(x["query_metric_keys"]) for x in records)) if records else set()
        if not set(labels.metric_key) <= covered:
            raise ValueError(f"Labels lack predictor provenance: {predictor}")


def confidence_fit(development_query_scores, development_top1_correct):
    """Development-only logistic calibration of top1 margin/entropy/coverage features."""
    from sklearn.linear_model import LogisticRegression
    x = np.asarray(development_query_scores, dtype=float)
    y = np.asarray(development_top1_correct, dtype=int)
    if len(np.unique(y)) != 2:
        raise ValueError("Confidence calibration needs both correct and incorrect queries")
    return LogisticRegression(C=.1, max_iter=1000).fit(x, y)


def query_confidence_features(candidate_scores):
    """Scale-relative margin, score entropy, top library evidence and candidate count."""
    x = np.sort(np.asarray(candidate_scores, dtype=float))[::-1]
    if not len(x) or not np.isfinite(x).all():
        return np.array([0., 1., 0.])
    z = (x-x.mean())/max(x.std(), 1e-6)
    p = np.exp(z-z.max()); p /= p.sum()
    entropy = -np.sum(p*np.log(np.maximum(p, 1e-12)))/max(np.log(len(x)), 1)
    return np.array([z[0]-z[1] if len(z)>1 else 0., entropy, np.log1p(len(x))])


def fit_nested_rankers(evidence, labels, split, provenance, destination, seeds=(2026, 31),
                       required_predictors=("dual",)):
    """Outer held-out MRR; inner development early stopping; listwise LightGBM ensemble.

    Features must be nested out-of-fold for EVERY base predictor, with an outer_fold column.
    Each outer panel uses predictors fitted without outer-test AND development labels.
    Merely grouping the
    final ranker cannot repair a leaked full-training FPNet/Transformer checkpoint.
    Missing truths are retained as zero-score queries in the reported denominator.
    """
    import lightgbm as lgb
    audit_feature_provenance(provenance, labels, split, required_predictors)
    if "outer_fold" not in evidence:
        raise ValueError("Nested evidence needs an outer_fold column; single OOF features are insufficient")
    dest = Path(destination); dest.mkdir(parents=True, exist_ok=True)
    if (dest/"nested_report.json").exists():
        raise FileExistsError("Use a fresh ranker experiment directory")
    query_meta = labels.merge(split[["metric_key", "fold", "group"]], on="metric_key", validate="many_to_one")
    if len(query_meta) != len(labels):
        raise ValueError("Some query labels are absent from the split manifest")
    f = evidence.merge(query_meta[["molecule_id", "metric_key", "fold"]].rename(columns={"metric_key": "truth"}),
        on="molecule_id", validate="many_to_one")
    f["target"] = f.metric_key.eq(f.truth).astype(int)
    provenance_coverage = {(r["predictor_id"], r["outer_fold"], k)
        for r in provenance for k in r["query_metric_keys"]}
    for r in f[["outer_fold", "truth"]].drop_duplicates().itertuples(index=False):
        for predictor in required_predictors:
            if (predictor, r.outer_fold, r.truth) not in provenance_coverage:
                raise ValueError(f"Missing nested feature provenance for {predictor}/{r.outer_fold}/{r.truth}")
    if f.duplicated(["outer_fold", "molecule_id", "metric_key"]).any():
        raise ValueError("Duplicate candidates in nested feature table")
    models, reports, oof = [], [], []
    folds = sorted(query_meta.fold.unique())
    if len(folds) < 3:
        raise ValueError("Nested fitting needs at least three outer folds")
    def xy(frame):
        z = frame.sort_values(["molecule_id", "metric_key"], kind="stable").copy()
        return z, z.reindex(columns=FEATURES).astype(float).replace([np.inf, -np.inf], np.nan), z.groupby("molecule_id", sort=False).size().to_numpy()
    for fi, outer in enumerate(folds):
        dev = folds[(fi+1)%len(folds)]
        panel = f[f.outer_fold.eq(outer)]
        tr, dv, ts = panel[~panel.fold.isin([outer, dev])], panel[panel.fold.eq(dev)], panel[panel.fold.eq(outer)]
        # Rank loss has no information for groups without a positive candidate.
        tr = tr[tr.molecule_id.isin(tr.groupby("molecule_id").target.max().loc[lambda s:s>0].index)]
        dv_fit = dv[dv.molecule_id.isin(dv.groupby("molecule_id").target.max().loc[lambda s:s>0].index)]
        if tr.empty or dv_fit.empty:
            raise ValueError("No positive candidate groups in training/development")
        a, ax, ag = xy(tr); b, bx, bg = xy(dv_fit); c, cx, _ = xy(ts)
        scores = []
        for seed in seeds:
            model = lgb.LGBMRanker(objective="lambdarank", label_gain=[0,1], n_estimators=1500,
                learning_rate=.025, num_leaves=15, min_child_samples=40, reg_lambda=10.,
                colsample_bytree=.85, subsample=.8, subsample_freq=1, random_state=seed,
                n_jobs=4, verbosity=-1, lambdarank_truncation_level=25)
            model.fit(ax, a.target, group=ag, eval_set=[(bx, b.target)], eval_group=[bg],
                eval_at=[1,5,25], callbacks=[lgb.early_stopping(80, verbose=False)])
            name = f"fold{outer}_seed{seed}.txt"; model.booster_.save_model(str(dest/name)); models.append(name)
            if len(c):
                scores.append(model.predict(cx))
        if len(c):
            c["score"] = np.mean(scores, axis=0)
            c = c.sort_values(["molecule_id", "score", "metric_key"], ascending=[True, False, True])
            c["final_rank"] = c.groupby("molecule_id").cumcount()+1
            c = c[c.final_rank<=25]; oof.append(c)
        else:
            c = pd.DataFrame(columns=["molecule_id", "metric_key", "final_rank"])
        reports.append(dict(outer_fold=int(outer), dev_fold=int(dev),
            **mrr25(c, query_meta[query_meta.fold.eq(outer)])))
    all_oof = pd.concat(oof, ignore_index=True) if oof else pd.DataFrame(columns=["molecule_id", "metric_key", "final_rank"])
    all_oof.to_parquet(dest/"outer_oof.parquet", index=False)
    report = dict(folds=reports, aggregate=mrr25(all_oof, labels),
                  note="NDCG surrogate trained; outer evaluation is actual MRR@25.")
    (dest/"nested_report.json").write_text(json.dumps(report, indent=2))
    (dest/"manifest.json").write_text(json.dumps(dict(features=FEATURES, models=models,
        training_scope="nested OOF base features; average outer models for final inference"), indent=2))
    return report


def paired_bootstrap(control_rr, challenger_rr, seed=2026, repetitions=5000):
    """Molecule-level paired interval; a small positive sample delta is not a guaranteed gain."""
    d = np.asarray(challenger_rr)-np.asarray(control_rr)
    if not len(d):
        raise ValueError("No held-out molecules")
    rng = np.random.default_rng(seed)
    samples = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(repetitions)])
    return dict(delta=float(d.mean()), ci95=np.quantile(samples, [.025,.975]).tolist(), molecules=len(d))
'''
(PACKAGE / 'validation.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/checkpoints.py
module_source = r'''"""Check a completed baseline against the actual query before reusing costly stages."""
from pathlib import Path
import json
import pandas as pd
import pyarrow as pa


REQUIRED = ["execution_receipt.json", "effective_query_spectra.parquet",
            "union_evidence.parquet", "submission_baseline0465.csv",
            "frozen_candidate_evidence.parquet", "control_neural.parquet",
            "submission_control.csv"]


def validate_completed_baseline(root, test, source_sha256):
    """Return a verified same-input checkpoint; reject partial or incompatible caches.

    Intended for the same immutable Kaggle input mounts after a late-stage failure.
    Every original query column/value must match, including fragment arrays and CE.
    The baseline's added neutral-mass column is ignored, not used as an identity shortcut.
    """
    root = Path(root)
    missing = [name for name in REQUIRED if not (root/name).is_file()]
    if missing:
        raise ValueError(f"Incomplete baseline checkpoint: {missing}; use a fresh session")
    receipt = json.loads((root/"execution_receipt.json").read_text())
    if receipt.get("source_sha256") != source_sha256 or not receipt.get("full_coverage_requested"):
        raise ValueError("Cached baseline source/full-coverage mode differs; use a fresh session")
    query = pd.read_parquet(root/"effective_query_spectra.parquet")
    if any(c not in query for c in test.columns):
        raise ValueError("Cached query is missing original test columns")
    def table(frame):
        return pa.Table.from_pandas(frame.reset_index(drop=True), preserve_index=False).replace_schema_metadata(None).combine_chunks()
    if not table(test).equals(table(query[test.columns])):
        raise ValueError("Cached query differs from actual test.parquet; use a fresh session")
    ids = set(test.molecule_id.astype(str))
    if receipt.get("processed_molecules") != len(ids):
        raise ValueError("Cached baseline did not cover every test molecule")
    union = pd.read_parquet(root/"union_evidence.parquet", columns=["molecule_id"])
    if set(union.molecule_id.astype(str)) != ids:
        raise ValueError("Cached candidate union has incomplete molecule coverage")
    from .ranking import validate_submission
    control = pd.read_csv(root/"submission_control.csv", dtype=str, keep_default_na=False)
    validate_submission(control, ids)
    return receipt
'''
(PACKAGE / 'checkpoints.py').write_text(module_source)


In [ ]:
# Module: casmi_refactor/output_contract.py
module_source = r'''"""Submission serialization, hidden-ID templates and artifact selection safeguards."""
from pathlib import Path
import json
import shutil
import zipfile
import pandas as pd
from .runtime import sha256


def ordered_test_ids(test, sample=None):
    """Use current test IDs; preserve sample order only if it covers exactly those IDs."""
    if test.empty or test.molecule_id.isna().any():
        raise ValueError("Empty test or null molecule_id")
    ids = test.molecule_id.astype(str).drop_duplicates().tolist()
    if sample is not None and "molecule_id" in sample and sample.molecule_id.notna().all():
        other = sample.molecule_id.astype(str).tolist()
        if len(other)==len(ids) and len(set(other))==len(other) and set(other)==set(ids):
            return other
    return ids


def make_runtime_competition(source, destination):
    """Prevent an unchanged visible sample from introducing wrong IDs during hidden inference.

    Only the template ID column is used by the inherited engines. Its empty SMILES values
    are never predictions. Train/test remain symlinks to the actual mounted inputs.
    """
    source, destination = Path(source).resolve(), Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    test = pd.read_parquet(source/"test.parquet", columns=["molecule_id"])
    sample = pd.read_csv(source/"sample_submission.csv", dtype=str) if (source/"sample_submission.csv").is_file() else None
    ids = ordered_test_ids(test, sample)
    pd.DataFrame({"molecule_id":ids,"smiles":[""]*len(ids)}).to_csv(destination/"sample_submission.csv", index=False)
    for name in ["train.parquet", "test.parquet"]:
        target = destination/name
        if target.exists() or target.is_symlink():
            if target.resolve() != (source/name).resolve():
                raise ValueError(f"Runtime input link points to different data: {target}")
        else:
            target.symlink_to(source/name)
    return destination


def export_verified(frame, test, path, sample=None):
    """Validate the serialized CSV itself, with exact current-test coverage and no index."""
    from .ranking import validate_submission
    ids = ordered_test_ids(test, sample)
    result = frame.loc[:,["molecule_id","smiles"]].copy()
    result["molecule_id"] = result.molecule_id.astype(str)
    validate_submission(result, ids)
    result = result.set_index("molecule_id").loc[ids].reset_index()
    path = Path(path); temp = path.with_suffix(".csv.tmp")
    try:
        result.to_csv(temp, index=False, encoding="utf-8", lineterminator="\n")
        readback = pd.read_csv(temp, dtype=str, keep_default_na=False)
        validate_submission(readback, ids)
        if not readback.equals(result):
            raise ValueError("Saved CSV differs from validated predictions")
        with temp.open("rb") as f:
            if f.readline().rstrip(b"\r\n") != b"molecule_id,smiles":
                raise ValueError("Serialized CSV header differs from competition schema")
        temp.replace(path)
    finally:
        if temp.exists():
            temp.unlink()
    return result


def archive_other_csvs(root):
    """Keep exactly one top-level CSV so a diagnostic CSV cannot be selected accidentally."""
    root = Path(root); artifacts = root/"diagnostics"; artifacts.mkdir(exist_ok=True)
    archive = artifacts/"csv_artifacts.zip"
    members = {}
    if archive.is_file():
        with zipfile.ZipFile(archive) as z:
            members = {name:z.read(name) for name in z.namelist()}
    paths = list(artifacts.glob("*.csv"))+[p for p in root.glob("*.csv") if p.name!="submission.csv"]
    runtime_sample = root/"runtime_competition/sample_submission.csv"
    if runtime_sample.is_file():
        paths.append(runtime_sample)
    moved = []
    for path in paths:
        name = ("runtime_sample_submission.csv" if path==runtime_sample else
                "earlier/"+path.name if path.parent==artifacts else path.name)
        members[name] = path.read_bytes(); moved.append(name)
    temp = archive.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temp,"w",zipfile.ZIP_DEFLATED) as z:
        for name,data in members.items():
            z.writestr(name,data)
    temp.replace(archive)
    # Every removed CSV has a verified byte-for-byte copy in the reversible archive.
    with zipfile.ZipFile(archive) as z:
        for path,name in zip(paths,moved):
            if z.read(name) != path.read_bytes():
                raise ValueError(f"Diagnostic archive verification failed: {path}")
        for path in paths:
            path.unlink()
    return moved


def restore_resume_csvs(root):
    """Restore only known baseline CSVs; full query/source checks still gate reuse."""
    root = Path(root)
    for name in ["submission_control.csv", "submission_baseline0465.csv"]:
        saved = root/"diagnostics"/name
        if not (root/name).exists() and saved.is_file():
            shutil.copy2(saved, root/name)
        archive = root/"diagnostics/csv_artifacts.zip"
        if not (root/name).exists() and archive.is_file():
            with zipfile.ZipFile(archive) as z:
                if name in z.namelist():
                    (root/name).write_bytes(z.read(name))


def final_certificate(root, test, mode, reason=None):
    """Print the exact file to choose and record a round-trip validation certificate."""
    root = Path(root); path = root/"submission.csv"
    result = pd.read_csv(path, dtype=str, keep_default_na=False)
    from .ranking import validate_submission
    validate_submission(result, test.molecule_id.astype(str).unique())
    moved = archive_other_csvs(root)
    counts = result.smiles.str.split(";").map(len)
    report = dict(selected_output="submission.csv", selected_absolute_path=str(path),
        submission_sha256=sha256(path), rows=len(result), unique_ids=int(result.molecule_id.nunique()),
        candidate_count_min=int(counts.min()), candidate_count_max=int(counts.max()),
        schema=result.columns.tolist(), nulls=int(result.isna().sum().sum()),
        actual_test_id_coverage=True, csv_round_trip_validated=True,
        final_mode=mode, fallback_reason=reason, archived_diagnostic_csvs=moved,
        leaderboard_score=None, grading_failure_root_cause="not supplied by Kaggle in the attached visible log")
    (root/"submission_certificate.json").write_text(json.dumps(report, indent=2))
    print("SELECT THIS COMPETITION OUTPUT: submission.csv", flush=True)
    print(json.dumps(report, indent=2), flush=True)
    return report


def with_control_fallback(operation, root, control, test, sample=None):
    """Attempt optional inference; explicitly recover a validated control on failure."""
    root = Path(root)
    try:
        return operation()
    except Exception as exc:
        import traceback
        (root/"challenger_failure.txt").write_text(traceback.format_exc())
        reason = f"{type(exc).__name__}: {exc}"
        print(f"CHALLENGER FAILED: {reason}; exporting verified control", flush=True)
        export_verified(control, test, root/"submission.csv", sample)
        report = final_certificate(root, test, "control_fallback", reason)
        (root/"refactor_report.json").write_text(json.dumps(report, indent=2))
        return report
'''
(PACKAGE / 'output_contract.py').write_text(module_source)


## Original ensemble and attribution

The following hash-checked archive contains the supplied frozen inference source and its original notices. Compression is limited to the inherited ensemble; the new modules above are readable. Submission mode changes its visible-test switch to process every molecule and exports overflow candidates. Existing trained feature semantics and baseline ranking rules are retained.

In [ ]:
import base64, io, zipfile
LEGACY_ARCHIVE_SHA256 = 'c53719fdbfaea5c127c5910dc5fcfbf82f2a1d39fa2114ad585b3119bf2e05ad'
LEGACY_ARCHIVE_B64 = '...'
archive_bytes = base64.b64decode(LEGACY_ARCHIVE_B64)
assert hashlib.sha256(archive_bytes).hexdigest() == LEGACY_ARCHIVE_SHA256
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for item in archive.infolist():
        target = (LEGACY / item.filename).resolve()
        assert target.is_relative_to(LEGACY.resolve())
    archive.extractall(LEGACY)
print("Verified and extracted", len(list(LEGACY.iterdir())), "inherited source files.")


## Packaging checks

These checks compile the complete refactor without executing external model code. They are engineering checks, not model validation.

In [ ]:
import ast
module_files = sorted(PACKAGE.glob("*.py"))
for file in module_files:
    ast.parse(file.read_text(), filename=str(file))
pins = json.loads((LEGACY / "pins.json").read_text())
assert hashlib.sha256((LEGACY / "frozen_0465.ipynb").read_bytes()).hexdigest() == pins["FROZEN_SHA256"]
print("Python modules parsed:", len(module_files))
print("Pinned original source:", pins["FROZEN_SHA256"])
print("Available dependencies:", {m: importlib.util.find_spec(m) is not None for m in ["numpy", "pandas", "pyarrow", "rdkit", "torch", "lightgbm"]})


## Regression verification

18 local tests passed, covering duplicate reference IDs, retrieval, training/checkpoint integration, changed query peaks, hidden IDs differing from the visible sample, CSV round-trip validation, artifact selection, and optional-stage fallback. These checks do not establish a Kaggle score. The following lightweight comparator check runs before expensive inference.


In [ ]:
# Early regression check: test the exact packaged heap entry without importing RDKit.
# This is safe before the pinned chemistry wheel is installed by submission inference.
import ast, heapq
from dataclasses import dataclass, field
import numpy as np
heap_tree = ast.parse((PACKAGE / "library.py").read_text())
heap_node = next(node for node in heap_tree.body if isinstance(node, ast.ClassDef) and node.name == "ReferenceEntry")
heap_scope = {"dataclass": dataclass, "field": field, "__name__": "__main__"}
exec(compile(ast.Module(body=[heap_node], type_ignores=[]), "heap_regression", "exec"), heap_scope)
Entry = heap_scope["ReferenceEntry"]
heap = []
for i in range(20):
    heapq.heappush(heap, Entry(0.5, "duplicate-id", "key", np.array([i, i+1.])))
for _ in range(5):
    heapq.heappop(heap)
for i in range(5):
    heapq.heapreplace(heap, Entry(0.6+i*.01, "replacement", "key", np.array([i, i+1.])))
assert len(heap) == 15
print("PASS: tied heap insertion, pop and replacement with NumPy array payloads")


## Configuration and inference

The scoring-repair default is `final_submission="control"`: the inherited final-25 neural recipe using weight 0.25 and offset 3.0. It skips expanded reference retrieval and challenger reranking. The original notebook's historical 0.466 is user-reported; this release has no new measured score.

The complete experimental challenger remains available by setting `final_submission="challenger"`. Its failures are recorded and the validated control is exported explicitly. Resume reuses completed work only after exact query/source checks. After success, `submission.csv` is the only top-level CSV; other CSV outputs are preserved inside `diagnostics/csv_artifacts.zip`.


In [ ]:
# Reload rewritten modules when recovering in an existing notebook session.
import importlib
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "casmi_refactor" or module_name.startswith("casmi_refactor."):
        del sys.modules[module_name]
from casmi_refactor.config import Config
CFG = Config(mode=MODE, output_dir=str(ROOT), final_submission="control")
CFG.validate()
print("Final submission mode:", CFG.final_submission)
print("Candidate union limit:", CFG.pool_limit)
print("New-candidate promotion limit:", CFG.max_new_in_top25)
if MODE in {"submission", "smoke"}:
    from casmi_refactor.pipeline import run_submission
    REPORT = run_submission(CFG, LEGACY)
else:
    print("Audit mode: full pretrained inference is not launched; no submission.csv is produced.")


## Local data audit

The local paths below point to the user-provided visible files. On another machine, change them or use the CLI. A visible-test audit is not an independent accuracy measurement.

In [ ]:
LOCAL_TEST = Path("/Users/aitzhanovdias/Downloads/test (5).parquet")
LOCAL_SAMPLE = Path("/Users/aitzhanovdias/Downloads/sample_submission (25).csv")
if MODE == "audit":
    needed = ["numpy", "pandas", "pyarrow", "rdkit"]
    missing = [m for m in needed if importlib.util.find_spec(m) is None]
    if missing:
        print("Data audit not executed; missing local dependencies:", missing)
        print("The complete inference code remains included. Use the configured Kaggle environment.")
    elif LOCAL_TEST.is_file() and LOCAL_SAMPLE.is_file():
        from casmi_refactor.audit import inspect_inputs
        AUDIT, CONSENSUS = inspect_inputs(LOCAL_TEST, LOCAL_SAMPLE, ROOT / "audit_results", CFG)
        print(json.dumps(AUDIT, indent=2))
        display(CONSENSUS.head(10))
    else:
        print("Set LOCAL_TEST and LOCAL_SAMPLE to run the data audit.")


## Separate model preparation and honest evaluation

Run these steps in a preparation environment with the full training data. Do not enable training in a nine-hour submission notebook. The complete implementations are in `training.py` and `validation.py` above.

```python
# PYTHONPATH must contain the directory holding casmi_refactor.
# CLI examples, executed outside this submission notebook:
# python -m casmi_refactor split --train train.parquet --output splits.parquet --folds 5
# python -m casmi_refactor prepare --train train.parquet --split splits.parquet --output store
# python -m casmi_refactor train --store store --output dual_fold4 \
#   --train-folds 0 1 2 --dev-folds 3 --test-folds 4
```

For each outer fold, refit every supervised base encoder without outer-test and development labels, and generate inner out-of-fold training evidence. Use `fit_nested_rankers(evidence, labels, split, provenance, destination, required_predictors=...)`; `evidence` must include `outer_fold`, and provenance must enumerate every active supervised predictor. This rejects ordinary single-OOF stacking as insufficient for strict nested validation. The original full-training pretrained bank cannot be relabeled fold-safe merely by grouping the final ranker.

`retrieve(..., excluded_keys=..., excluded_scaffolds=...)` supports class-2 replay without held-out spectral references. Class-1 library retrieval may deliberately use independently acquired reference spectra of the same structure, while the encoder fitting split still excludes the chemical group. Do not reuse identical scans as independent evidence. Class-3 evaluation must additionally remove the answer structure from all structure catalogs and candidate seeds; this package does not claim to have measured class-3 de novo performance.

Report candidate recall before ranking, top-1, MRR@25, and paired molecule-bootstrap intervals. Missing candidate truths remain zero in the MRR denominator. Fit `confidence_fit` only on development query margin/entropy features; raw RRF, fingerprint logits and simulator scores are not confidence probabilities. New policy weights need nested development selection and a locked outer test before claiming improvement.

A freshly trained optional checkpoint can be activated with `Config(dual_checkpoint=".../model.pt")`. Optional learned rankers use `Config(ranker_file=".../ranker_directory")`. Otherwise the inherited pretrained ensemble and transparent heuristic challenger run without a new checkpoint. FAISS is optional for continuous embedding retrieval; exact mass-gated NumPy search is the fallback. Binary Hamming search is not presented as Tanimoto.


## Review findings and limits

- The supplied execution log processed 12/400 molecules and reported 388 placeholders. Its own hidden rerun path intended full coverage; this was a visible smoke shortcut, not proof the scored hidden run used placeholders.
- The supplied final reranker explicitly used an unvalidated fixed neural weight of 0.25 and offset 3.0. It could not retrieve a truth absent from its frozen final candidates.
- The original pipeline already contains multiple fingerprint backbones, PubChem/COCONUT retrieval, generated candidates, LightGBM rankers, ICEBERG and GLACIER. Replacing them with an untrained Transformer would discard valuable trained components. They are retained as the inference foundation.
- The new streaming rescue is approximate when its per-query reservoir cap is hit; sampling counts are reported. It does not claim Flash Entropy Search's indexing speed.
- Per-molecule mass consensus checks reported adduct consistency. Absolute precursor correction defaults to zero without independent lock-mass evidence. Charged/protomeric candidates receive a soft mass feature rather than a destructive formula veto.
- New default fusion weights are hypotheses and can reduce MRR. The control file allows direct ablation. A full hidden-test runtime, score increase, and end-to-end fold-safe evaluation remain unmeasured because full training data and original model assets were not provided locally.

## Primary references

The competition pages were requested, but their JavaScript pages did not expose body text to the research tool; the supplied competition description and notebook were used for the detailed contract.

- [Competition overview](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/overview)
- [Organizer metric page](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank)
- [Spectral entropy similarity](https://www.nature.com/articles/s41592-021-01331-z)
- [Flash entropy search](https://pmc.ncbi.nlm.nih.gov/articles/PMC11511675/)
- [MIST implementation](https://github.com/coleygroup/mist)
- [ICEBERG/GLACIER source and checkpoints](https://github.com/coleygroup/ms-pred)
- [DreaMS paper](https://www.nature.com/articles/s41587-025-02663-3)
- [LightGBM ranking objectives](https://github.com/lightgbm-org/LightGBM/blob/main/docs/Parameters.rst)
- [FAISS binary-index semantics](https://github.com/facebookresearch/faiss/wiki/Binary-Indexes)

Original notices and source licenses are preserved in the embedded frozen notebook; external datasets retain their own terms.


## Competition output selection

For a new competition submission, select the completed version's **`submission.csv`**. Baseline/reference CSVs are stored in `diagnostics/csv_artifacts.zip`. `submission_certificate.json` records validation results. The certificate is not itself a submission.


In [ ]:
if MODE == "submission":
    final_path = ROOT / "submission.csv"
    certificate = json.loads((ROOT / "submission_certificate.json").read_text())
    assert final_path.is_file() and certificate["actual_test_id_coverage"]
    assert certificate["candidate_count_max"] <= 25
    assert sorted(p.name for p in ROOT.glob("*.csv")) == ["submission.csv"]
    assert hashlib.sha256(final_path.read_bytes()).hexdigest() == certificate["submission_sha256"]
    print("SELECT THIS COMPETITION OUTPUT: submission.csv")
    print("Validated rows:", certificate["rows"], "| final mode:", certificate["final_mode"])
    from IPython.display import FileLink, display
    display(FileLink(str(final_path)))
else:
    print("Local audit only: no competition submission.csv was generated.")
